# Rooftop LiDAR → georeferenced, segmented, solar-ready model

Takes an iPhone/ARKit rooftop scan plus a co-registered DSM / RGB / building-mask
triple and produces a **georeferenced, semantically segmented** roof model with
per-surface pitch, azimuth, elevation and usable area.

**Pipeline**

| Stage | What it does |
|---|---|
| 1 | Load and characterise the four inputs |
| 2 | Solve the ARKit → UTM transform (4-DOF: yaw, E, N, Z + optional scale) |
| 3 | Segment into planes, terraces and objects |
| 4 | Derive solar attributes: tilt, true azimuth, usable area, shading |
| 5 | Export in local-ENU **and** absolute UTM, plus GeoJSON / CSV / JSON |

**Three things this pipeline is careful about**, each of which silently ruins the
output if you get it wrong:

1. **Chirality.** ARKit is `x` right, `y` up, `z` toward viewer. Mapping to ENU as
   `east=+x, north=−z, up=+y` preserves handedness. Using `north=+z` mirrors the
   model and flips every azimuth. Stage 2 *proves* the choice against the data.
2. **The 180° ambiguity.** A roughly rectangular footprint matches its own
   180° rotation almost perfectly. Binary overlap cannot separate them; the
   superstructure **height** map can. Getting this wrong turns south-facing into
   north-facing.
3. **Grid vs true north.** UTM azimuths are grid azimuths. Solar geometry needs
   true north. Grid convergence is applied to every azimuth that leaves the pipeline.

## 0 · Environment

In [ ]:
import sys, subprocess, importlib

# find_spec("google.colab") imports the parent package and raises off-Colab,
# so probe by import instead.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

# rasterio is the only thing Colab lacks; numpy/scipy/matplotlib are preinstalled.
for pkg, mod in [("rasterio", "rasterio")]:
    if importlib.util.find_spec(mod) is None:
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np, scipy, matplotlib
print("numpy", np.__version__, "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)
try:
    import rasterio; print("rasterio", rasterio.__version__)
except ImportError:
    from osgeo import gdal; print("using GDAL", gdal.__version__)

## 1 · Mount Drive and point at the inputs

Expected Drive layout — create this folder and upload the four files into it:

```
MyDrive/
└── iphone-lidar/
    └── inputs/
        ├── lidar_object.zip
        ├── dsm.tif
        ├── rgb.tif
        └── building_mask.tif
```

Outputs are written next to it in `MyDrive/iphone-lidar/outputs/`. Running
outside Colab falls back to local paths, so the same notebook works on a laptop.

In [ ]:
import os, pathlib

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # opens a Google auth prompt
    BASE = pathlib.Path("/content/drive/MyDrive/iphone-lidar")
else:
    BASE = pathlib.Path(os.environ.get("ROOFLIB_BASE", pathlib.Path.cwd()))

INPUT_DIR  = BASE / "inputs"
OUTPUT_DIR = BASE / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED = ["lidar_object.zip", "dsm.tif", "rgb.tif", "building_mask.tif"]

print("input dir :", INPUT_DIR)
print("output dir:", OUTPUT_DIR)

# Convenience: if the four files are absent but the single-file bundle is
# sitting in Drive, unpack it. Uploading one 21 MB zip is much less fiddly
# than four separate files.
if any(not (INPUT_DIR / f).exists() for f in REQUIRED):
    for cand in [BASE / "iphone-lidar-inputs.zip", INPUT_DIR / "iphone-lidar-inputs.zip"]:
        if cand.exists():
            import zipfile
            print(f"unpacking bundle {cand.name} ...")
            INPUT_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(cand) as z:
                z.extractall(INPUT_DIR)
            break

missing = [f for f in REQUIRED if not (INPUT_DIR / f).exists()]
if missing:
    raise SystemExit(
        f"Missing {missing} in {INPUT_DIR}.\n"
        "Either upload those files there, or drop iphone-lidar-inputs.zip into\n"
        f"{BASE} and re-run this cell."
    )
for f in REQUIRED:
    print(f"  {f:24s} {(INPUT_DIR / f).stat().st_size / 1e6:8.2f} MB")

## 2 · Materialise `rooflib`

Written to disk here so the notebook stays self-contained — nothing to upload
besides the inputs, and nothing to `pip install` from a private index.

In [ ]:
import pathlib
pathlib.Path('rooflib').mkdir(exist_ok=True)
pathlib.Path('rooflib/__init__.py').write_text('')
print('rooflib package created')

In [ ]:
%%writefile rooflib/io_utils.py
"""Input loading for the rooftop pipeline.

Deliberately dependency-light: rasters go through a shim that prefers rasterio
(what Colab gets from `pip install rasterio`) and falls back to osgeo/GDAL.
The OBJ reader is hand-rolled numpy so the pipeline does not need trimesh or
open3d, neither of which is preinstalled anywhere we run.
"""
from __future__ import annotations

import io
import json
import zipfile
from dataclasses import dataclass, field

import numpy as np


# --------------------------------------------------------------------------
# rasters
# --------------------------------------------------------------------------
@dataclass
class Raster:
    """A georeferenced grid. `transform` is the 6-tuple GDAL geotransform."""

    data: np.ndarray  # (H, W) or (B, H, W)
    transform: tuple
    crs_wkt: str
    nodata: float | None = None

    @property
    def shape(self):
        return self.data.shape[-2:]

    @property
    def res(self):
        return abs(self.transform[1]), abs(self.transform[5])

    def xy(self, rows, cols, center=True):
        """Pixel indices -> world (easting, northing)."""
        gt = self.transform
        off = 0.5 if center else 0.0
        e = gt[0] + (np.asarray(cols) + off) * gt[1] + (np.asarray(rows) + off) * gt[2]
        n = gt[3] + (np.asarray(cols) + off) * gt[4] + (np.asarray(rows) + off) * gt[5]
        return e, n

    def rowcol(self, e, n):
        """World (easting, northing) -> fractional pixel indices."""
        gt = self.transform
        det = gt[1] * gt[5] - gt[2] * gt[4]
        de, dn = np.asarray(e) - gt[0], np.asarray(n) - gt[3]
        col = (de * gt[5] - dn * gt[2]) / det
        row = (dn * gt[1] - de * gt[4]) / det
        return row - 0.5, col - 0.5

    def sample(self, e, n, order=1):
        """Bilinear sample of a single-band raster at world coordinates.

        Returns NaN outside the grid so callers can mask cleanly.
        """
        arr = self.data if self.data.ndim == 2 else self.data[0]
        r, c = self.rowcol(e, n)
        h, w = arr.shape
        if order == 0:
            ri, ci = np.round(r).astype(int), np.round(c).astype(int)
            ok = (ri >= 0) & (ri < h) & (ci >= 0) & (ci < w)
            out = np.full(np.shape(e), np.nan, float)
            out[ok] = arr[ri[ok], ci[ok]]
            return out
        r0 = np.floor(r).astype(int)
        c0 = np.floor(c).astype(int)
        fr, fc = r - r0, c - c0
        out = np.full(np.shape(e), np.nan, float)
        ok = (r0 >= 0) & (r0 + 1 < h) & (c0 >= 0) & (c0 + 1 < w)
        if not np.any(ok):
            return out
        r0o, c0o, fro, fco = r0[ok], c0[ok], fr[ok], fc[ok]
        v = (
            arr[r0o, c0o] * (1 - fro) * (1 - fco)
            + arr[r0o + 1, c0o] * fro * (1 - fco)
            + arr[r0o, c0o + 1] * (1 - fro) * fco
            + arr[r0o + 1, c0o + 1] * fro * fco
        )
        out[ok] = v
        return out


def read_raster(path) -> Raster:
    """Read a GeoTIFF via rasterio if present, else osgeo."""
    try:
        import rasterio

        with rasterio.open(path) as ds:
            data = ds.read()
            gt = ds.transform.to_gdal()
            wkt = ds.crs.to_wkt() if ds.crs else ""
            nod = ds.nodata
    except ImportError:
        from osgeo import gdal

        gdal.UseExceptions()
        ds = gdal.Open(str(path))
        data = ds.ReadAsArray()
        gt = ds.GetGeoTransform()
        wkt = ds.GetProjection()
        nod = ds.GetRasterBand(1).GetNoDataValue()
    if data.ndim == 3 and data.shape[0] == 1:
        data = data[0]
    return Raster(np.asarray(data), tuple(gt), wkt, nod)


def write_raster(path, arr, transform, crs_wkt, nodata=None):
    """Write a single- or multi-band GeoTIFF (rasterio preferred)."""
    arr = np.asarray(arr)
    bands = 1 if arr.ndim == 2 else arr.shape[0]
    a3 = arr[None] if arr.ndim == 2 else arr
    try:
        import rasterio
        from rasterio.transform import Affine

        tr = Affine.from_gdal(*transform)
        with rasterio.open(
            path, "w", driver="GTiff", height=a3.shape[1], width=a3.shape[2],
            count=bands, dtype=a3.dtype, crs=crs_wkt or None, transform=tr,
            nodata=nodata, compress="deflate",
        ) as ds:
            ds.write(a3)
    except ImportError:
        from osgeo import gdal, gdal_array

        gdal.UseExceptions()
        drv = gdal.GetDriverByName("GTiff")
        gtype = gdal_array.NumericTypeCodeToGDALTypeCode(a3.dtype.type)
        ds = drv.Create(str(path), a3.shape[2], a3.shape[1], bands, gtype,
                        options=["COMPRESS=DEFLATE"])
        ds.SetGeoTransform(transform)
        ds.SetProjection(crs_wkt)
        for i in range(bands):
            b = ds.GetRasterBand(i + 1)
            b.WriteArray(a3[i])
            if nodata is not None:
                b.SetNoDataValue(float(nodata))
        ds.FlushCache()


# --------------------------------------------------------------------------
# mesh
# --------------------------------------------------------------------------
@dataclass
class Mesh:
    V: np.ndarray            # (nv, 3) vertices, ARKit frame (x right, y up, z toward viewer)
    F: np.ndarray            # (nf, 3) triangle vertex indices
    VT: np.ndarray | None = None   # (nvt, 2) texture coords
    FT: np.ndarray | None = None   # (nf, 3) texture indices
    face_material: np.ndarray | None = None   # (nf,) material name per face
    face_group: np.ndarray | None = None      # (nf,) `o` group per face
    materials: dict = field(default_factory=dict)  # name -> texture relative path

    # -- derived quantities -------------------------------------------------
    def triangles(self):
        return self.V[self.F[:, 0]], self.V[self.F[:, 1]], self.V[self.F[:, 2]]

    def face_normals(self, normalize=True):
        p0, p1, p2 = self.triangles()
        n = np.cross(p1 - p0, p2 - p0)
        if not normalize:
            return n
        ln = np.linalg.norm(n, axis=1, keepdims=True)
        return n / np.maximum(ln, 1e-12)

    def face_areas(self):
        p0, p1, p2 = self.triangles()
        return np.linalg.norm(np.cross(p1 - p0, p2 - p0), axis=1) / 2.0

    def face_centroids(self):
        p0, p1, p2 = self.triangles()
        return (p0 + p1 + p2) / 3.0

    def sample_surface(self, n_per_face=10, seed=0, include_vertices=True):
        """Barycentric point sampling so top-down rasters come out solid."""
        p0, p1, p2 = self.triangles()
        rng = np.random.default_rng(seed)
        out = [self.V] if include_vertices else []
        for _ in range(n_per_face):
            a = rng.random((len(self.F), 1))
            b = rng.random((len(self.F), 1))
            flip = (a + b) > 1
            a = np.where(flip, 1 - a, a)
            b = np.where(flip, 1 - b, b)
            out.append(p0 + a * (p1 - p0) + b * (p2 - p0))
        return np.vstack(out)

    def sample_surface_with_normals(self, n_per_face=10, seed=0):
        """Points plus the normal and area-weight of the face each came from."""
        p0, p1, p2 = self.triangles()
        nrm = self.face_normals()
        area = self.face_areas()
        rng = np.random.default_rng(seed)
        pts, nn, ww, ff = [], [], [], []
        idx = np.arange(len(self.F))
        for _ in range(n_per_face):
            a = rng.random((len(self.F), 1))
            b = rng.random((len(self.F), 1))
            flip = (a + b) > 1
            a = np.where(flip, 1 - a, a)
            b = np.where(flip, 1 - b, b)
            pts.append(p0 + a * (p1 - p0) + b * (p2 - p0))
            nn.append(nrm)
            ww.append(area / n_per_face)
            ff.append(idx)
        return (np.vstack(pts), np.vstack(nn),
                np.concatenate(ww), np.concatenate(ff))

    def connected_shells(self):
        """Union-find over shared vertices -> per-face shell id.

        This mesh is heavily fragmented, so the result is informational; do not
        rely on it for segmentation.
        """
        parent = np.arange(len(self.V))

        def find(a):
            while parent[a] != a:
                parent[a] = parent[parent[a]]
                a = parent[a]
            return a

        for f in self.F:
            for x, y in ((f[0], f[1]), (f[1], f[2])):
                ra, rb = find(x), find(y)
                if ra != rb:
                    parent[ra] = rb
        roots = np.array([find(i) for i in range(len(self.V))])
        _, inv = np.unique(roots, return_inverse=True)
        return inv[self.F[:, 0]]


def load_obj_from_zip(zip_path, obj_name=None) -> tuple[Mesh, dict]:
    """Read the OBJ (and its MTL) straight out of the zip, no extraction."""
    zf = zipfile.ZipFile(zip_path)
    names = zf.namelist()
    if obj_name is None:
        cands = [n for n in names if n.lower().endswith(".obj")]
        if not cands:
            raise FileNotFoundError(f"no .obj inside {zip_path}")
        obj_name = cands[0]
    text = zf.read(obj_name).decode("utf-8", "replace")

    V, VT, F, FT = [], [], [], []
    fmat, fgrp = [], []
    cur_mat, cur_grp, mtllib = None, None, None
    for line in text.splitlines():
        if not line or line[0] == "#":
            continue
        tag, _, rest = line.partition(" ")
        if tag == "v":
            p = rest.split()
            V.append((float(p[0]), float(p[1]), float(p[2])))
        elif tag == "vt":
            p = rest.split()
            VT.append((float(p[0]), float(p[1])))
        elif tag == "f":
            toks = rest.split()
            vi, ti = [], []
            for t in toks:
                parts = t.split("/")
                vi.append(int(parts[0]) - 1)
                ti.append(int(parts[1]) - 1 if len(parts) > 1 and parts[1] else -1)
            # fan-triangulate any n-gons
            for k in range(1, len(vi) - 1):
                F.append((vi[0], vi[k], vi[k + 1]))
                FT.append((ti[0], ti[k], ti[k + 1]))
                fmat.append(cur_mat)
                fgrp.append(cur_grp)
        elif tag == "o" or tag == "g":
            cur_grp = rest.strip()
        elif tag == "usemtl":
            cur_mat = rest.strip()
        elif tag == "mtllib":
            mtllib = rest.strip()

    materials = {}
    if mtllib and mtllib in names:
        cur = None
        for line in zf.read(mtllib).decode("utf-8", "replace").splitlines():
            p = line.split()
            if not p:
                continue
            if p[0] == "newmtl":
                cur = p[1]
            elif p[0] == "map_Kd" and cur:
                materials[cur] = " ".join(p[1:])

    mesh = Mesh(
        V=np.asarray(V, float),
        F=np.asarray(F, np.int64),
        VT=np.asarray(VT, float) if VT else None,
        FT=np.asarray(FT, np.int64) if FT else None,
        face_material=np.asarray(fmat, object),
        face_group=np.asarray(fgrp, object),
        materials=materials,
    )
    meta = {
        "zip": str(zip_path),
        "obj": obj_name,
        "mtl": mtllib,
        "textures": {k: v for k, v in materials.items()},
        "n_vertices": int(len(mesh.V)),
        "n_faces": int(len(mesh.F)),
    }
    return mesh, meta


def load_textures_from_zip(zip_path, materials, max_side=None):
    """Decode texture JPEGs into numpy arrays (only needed for ortho render)."""
    from PIL import Image

    Image.MAX_IMAGE_PIXELS = None
    zf = zipfile.ZipFile(zip_path)
    out = {}
    for name, rel in materials.items():
        if rel not in zf.namelist():
            continue
        im = Image.open(io.BytesIO(zf.read(rel))).convert("RGB")
        if max_side and max(im.size) > max_side:
            sc = max_side / max(im.size)
            im = im.resize((int(im.width * sc), int(im.height * sc)), Image.BILINEAR)
        out[name] = np.asarray(im)
    return out


def save_json(path, obj):
    def default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        raise TypeError(type(o))

    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, default=default)

In [ ]:
%%writefile rooflib/georef.py
"""Georeference the ARKit mesh onto the DSM/mask rasters.

The problem is 4-DOF, not 7-DOF: ARKit's +Y is gravity-up (measured here to
0.66 deg), and ARKit is metric. So we solve yaw, translation (E, N, Z) and
optionally a small scale correction.

Two failure modes drive the design:

1. The footprint is nearly point-symmetric, so binary overlap cannot tell
   yaw from yaw+180 (0.978 vs 0.972 on this dataset -- useless). The
   superstructure HEIGHT map separates them cleanly (+0.844 vs -0.434).
   `solve_yaw` therefore always scores both channels and reports the margin.

2. Mapping ARKit to ENU with the wrong chirality silently mirrors the model.
   `AXIS_MAP` below is the right-handed choice and `handedness_check` proves
   it against the data rather than assuming it.
"""
from __future__ import annotations

import numpy as np

# ARKit (x right, y up, z toward viewer) -> ENU-before-yaw (east, north, up).
# east = +x, north = -z, up = +y.  Determinant +1, so chirality is preserved.
AXIS_MAP = np.array([[1.0, 0.0, 0.0],
                     [0.0, 0.0, -1.0],
                     [0.0, 1.0, 0.0]])


def estimate_deck_level(mesh, bin_size=0.25, up_thresh=0.98):
    """Find the walking surface in the ARKit frame.

    On a rooftop scan the deck is the *lowest* large horizontal surface --
    everything above it is parapet, stair headroom and tanks. Picking the
    largest horizontal band alone is not enough on buildings with a big upper
    terrace, so we take the largest band and then prefer any comparably large
    band below it.
    """
    N = mesh.face_normals()
    A = mesh.face_areas()
    C = mesh.face_centroids()
    up = N[:, 1] > up_thresh
    if up.sum() == 0:
        raise ValueError("no horizontal faces found")
    y = C[up, 1]
    w = A[up]
    lo, hi = y.min(), y.max()
    edges = np.arange(lo, hi + bin_size, bin_size)
    hist, _ = np.histogram(y, bins=edges, weights=w)
    biggest = hist.max()
    idx = [i for i, v in enumerate(hist) if v >= 0.35 * biggest]
    pick = min(idx)  # lowest band that is still a major surface
    sel = (y >= edges[pick] - bin_size) & (y < edges[pick + 1] + bin_size)
    deck = float(np.average(y[sel], weights=w[sel]))
    return deck, {
        "deck_y_local": deck,
        "deck_area_m2": float(w[sel].sum()),
        "bands": [{"y": float(edges[i]), "area_m2": float(hist[i])}
                  for i in range(len(hist)) if hist[i] > 0.5],
    }


def deck_normal(mesh, deck_y, tol=0.15, up_thresh=0.98):
    """Area-weighted normal of the deck -- our gravity witness."""
    N = mesh.face_normals()
    A = mesh.face_areas()
    C = mesh.face_centroids()
    sel = (N[:, 1] > up_thresh) & (np.abs(C[:, 1] - deck_y) < tol)
    n = (A[sel][:, None] * N[sel]).sum(0)
    n /= np.linalg.norm(n)
    return n, float(np.degrees(np.arccos(np.clip(n[1], -1, 1)))), float(A[sel].sum())


def rot_z(deg):
    t = np.radians(deg)
    c, s = np.cos(t), np.sin(t)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])


def build_matrix(yaw_deg, scale, translation, pivot=(0.0, 0.0, 0.0),
                 tilt_correction=None):
    """Compose the 4x4 ARKit -> ENU transform.

        p_enu = scale * Rz(yaw) @ (AXIS_MAP @ p_arkit - pivot) + translation

    `pivot` lives in the post-AXIS_MAP, pre-yaw frame: (mean east, mean north,
    deck height) of the mesh. Solving around it keeps yaw and translation
    decoupled during optimisation, and makes `translation` directly readable
    as "where the deck centre sits in the world".
    """
    R = rot_z(yaw_deg) @ AXIS_MAP
    if tilt_correction is not None:
        R = tilt_correction @ R
    M = np.eye(4)
    M[:3, :3] = scale * R
    M[:3, 3] = np.asarray(translation, float) - scale * (rot_z(yaw_deg) @ np.asarray(pivot, float))
    return M


def apply(M, pts):
    return pts @ M[:3, :3].T + M[:3, 3]


def apply_normals(M, normals):
    """Transform normals.

    Points here use the row-vector convention (`apply` does `pts @ A.T`), so a
    normal maps as  n_row -> n_row @ inv(A)  -- NOT `@ inv(A).T`, which applies
    the inverse rotation and silently mirrors every azimuth.
    """
    A = M[:3, :3]
    N = normals @ np.linalg.inv(A)
    return N / np.maximum(np.linalg.norm(N, axis=1, keepdims=True), 1e-12)


# --------------------------------------------------------------------------
# target extraction
# --------------------------------------------------------------------------
def build_target(dsm, mask, drop_below=2.0):
    """The real building top surface.

    The supplied mask is a coarse rotated rectangle that overshoots the roof
    (on this dataset ~52 of 201.6 m2 sits ~11.7 m below the roof). Cutting at
    roof_median - drop_below removes that and leaves the surface the mesh can
    actually be matched against.
    """
    D = dsm.data.astype(float)
    M = mask.data
    inside = M == 1
    roof_median = float(np.median(D[inside]))
    building = inside & (D > roof_median - drop_below)
    return {
        "building": building,
        "mask_inside": inside,
        "roof_median": roof_median,
        "mask_area_m2": float(inside.sum() * dsm.res[0] * dsm.res[1]),
        "building_area_m2": float(building.sum() * dsm.res[0] * dsm.res[1]),
        "discarded_area_m2": float((inside & ~building).sum() * dsm.res[0] * dsm.res[1]),
    }


# --------------------------------------------------------------------------
# rasterisation helpers
# --------------------------------------------------------------------------
def _height_raster(e, n, z, origin_e, origin_n, res, size):
    """Top-down max-height raster; empty cells are NaN."""
    gi = np.floor((origin_n - n) / res).astype(int) + size // 2
    gj = np.floor((e - origin_e) / res).astype(int) + size // 2
    ok = (gi >= 0) & (gi < size) & (gj >= 0) & (gj < size)
    img = np.full((size, size), -1e9)
    np.maximum.at(img, (gi[ok], gj[ok]), z[ok])
    out = np.where(img > -1e8, img, np.nan)
    return out


def _pre_enu(mesh_pts):
    """ARKit -> ENU axes, before yaw. Returns (e0, n0, up)."""
    q = mesh_pts @ AXIS_MAP.T
    return q[:, 0], q[:, 1], q[:, 2]


def _transformed(e0, n0, up, yaw, scale, deck_up):
    t = np.radians(yaw)
    c, s = np.cos(t), np.sin(t)
    return scale * (c * e0 - s * n0), scale * (s * e0 + c * n0), scale * (up - deck_up)


# --------------------------------------------------------------------------
# coarse solve
# --------------------------------------------------------------------------
def solve_yaw(mesh_pts, target, dsm, deck_up, res=0.2, size=192,
              yaw_step=1.0, scale=1.0, mirror=False, height_clip=4.0):
    """Sweep yaw; at each yaw solve translation in closed form by FFT.

    Returns a list of dicts sorted by binary overlap, each carrying both the
    overlap and the height correlation so the 180 deg pair can be separated.
    """
    building = target["building"]
    ys, xs = np.nonzero(building)
    te, tn = dsm.xy(ys, xs)
    tz = dsm.data.astype(float)[ys, xs] - target["roof_median"]
    oe, on = te.mean(), tn.mean()

    T = _height_raster(te, tn, tz, oe, on, res, size)
    Tm = np.isfinite(T)
    Tv = np.where(Tm, np.clip(np.nan_to_num(T), 0, height_clip), 0.0)
    FTm = np.fft.rfft2(Tm.astype(float))

    e0, n0, up = _pre_enu(mesh_pts)
    e0 = e0 - e0.mean()
    n0 = n0 - n0.mean()
    if mirror:
        n0 = -n0

    out = []
    for yaw in np.arange(0.0, 360.0, yaw_step):
        re, rn, rz = _transformed(e0, n0, up, yaw, scale, deck_up)
        S = _height_raster(re, rn, rz, 0.0, 0.0, res, size)
        Sm = np.isfinite(S)
        Sv = np.where(Sm, np.clip(np.nan_to_num(S), 0, height_clip), 0.0)

        corr = np.fft.irfft2(np.fft.rfft2(Sm.astype(float)) * np.conj(FTm), s=T.shape)
        k = int(np.argmax(corr))
        di, dj = np.unravel_index(k, T.shape)
        overlap = corr.flat[k] / max(min(Sm.sum(), Tm.sum()), 1)

        S2 = np.roll(np.roll(Sv, -di, 0), -dj, 1)
        S2m = np.roll(np.roll(Sm, -di, 0), -dj, 1)
        both = S2m & Tm
        if both.sum() > 30 and S2[both].std() > 1e-6 and Tv[both].std() > 1e-6:
            hcorr = float(np.corrcoef(S2[both], Tv[both])[0, 1])
            hrmse = float(np.sqrt(np.mean((S2[both] - Tv[both]) ** 2)))
        else:
            hcorr, hrmse = float("nan"), float("nan")

        dE = (dj if dj <= size // 2 else dj - size) * res
        dN = -(di if di <= size // 2 else di - size) * res
        # translation that maps the (mean-centred, yaw-rotated) mesh onto target
        out.append({
            "yaw": float(yaw), "overlap": float(overlap),
            "hcorr": hcorr, "hrmse": hrmse,
            "tE": float(oe + dE), "tN": float(on + dN),
        })
    out.sort(key=lambda d: -d["overlap"])
    return out, {"origin_e": oe, "origin_n": on}


def joint_score(cand):
    """Rank on footprint AND height together.

    Overlap alone cannot see a 180 deg flip; height correlation alone can be
    fooled by a pose that stacks the superstructures plausibly but puts the
    outline in the wrong place. The product only rewards agreeing on both.
    """
    h = cand["hcorr"]
    if not np.isfinite(h):
        h = -1.0
    return cand["overlap"] * max(h, 0.0)


def handedness_check(mesh_pts, target, dsm, deck_up, **kw):
    """Prove the axis mapping empirically instead of trusting the convention."""
    res = {}
    for mirror in (False, True):
        cand, _ = solve_yaw(mesh_pts, target, dsm, deck_up, mirror=mirror, **kw)
        best = max(cand, key=joint_score)
        best = dict(best, joint=joint_score(best))
        res["mirrored" if mirror else "right_handed"] = best
    rh, mi = res["right_handed"], res["mirrored"]
    res["verdict"] = "right_handed" if rh["joint"] > mi["joint"] else "mirrored"
    res["margin"] = float(rh["joint"] - mi["joint"])
    return res


def disambiguate_180(candidates):
    """Pick between yaw and yaw+180 using the height channel."""
    best = max(candidates, key=lambda d: d["overlap"])
    opp = min(candidates, key=lambda d: abs(((d["yaw"] - best["yaw"]) % 360) - 180))
    winner = best if (best["hcorr"] or -9) >= (opp["hcorr"] or -9) else opp
    return winner, {
        "primary": {k: best[k] for k in ("yaw", "overlap", "hcorr")},
        "opposite": {k: opp[k] for k in ("yaw", "overlap", "hcorr")},
        "overlap_margin": float(best["overlap"] - opp["overlap"]),
        "hcorr_margin": float(abs((best["hcorr"] or 0) - (opp["hcorr"] or 0))),
        "decided_by": "height_correlation",
        "chosen_yaw": float(winner["yaw"]),
    }


# --------------------------------------------------------------------------
# scale (symmetric metric -- an overlap ratio normalised by min() is biased
# toward shrinking the source and must not be used here)
# --------------------------------------------------------------------------
def scale_iou(mesh_pts, target, dsm, deck_up, yaw, scales, res=0.2, size=192):
    building = target["building"]
    ys, xs = np.nonzero(building)
    te, tn = dsm.xy(ys, xs)
    tz = dsm.data.astype(float)[ys, xs] - target["roof_median"]
    oe, on = te.mean(), tn.mean()
    T = _height_raster(te, tn, tz, oe, on, res, size)
    Tm = np.isfinite(T)
    FTm = np.fft.rfft2(Tm.astype(float))

    e0, n0, up = _pre_enu(mesh_pts)
    e0, n0 = e0 - e0.mean(), n0 - n0.mean()

    rows = []
    for sc in scales:
        re, rn, rz = _transformed(e0, n0, up, yaw, sc, deck_up)
        S = _height_raster(re, rn, rz, 0.0, 0.0, res, size)
        Sm = np.isfinite(S)
        corr = np.fft.irfft2(np.fft.rfft2(Sm.astype(float)) * np.conj(FTm), s=T.shape)
        k = int(np.argmax(corr))
        di, dj = np.unravel_index(k, T.shape)
        S2m = np.roll(np.roll(Sm, -di, 0), -dj, 1)
        inter = float((S2m & Tm).sum())
        union = float((S2m | Tm).sum())
        rows.append({"scale": float(sc), "iou": inter / max(union, 1),
                     "intersection_px": inter, "union_px": union})
    return rows


# --------------------------------------------------------------------------
# refinement
# --------------------------------------------------------------------------
def refine(mesh_pts, mesh_normals_pt, target, dsm, deck_up, yaw0, tE0, tN0,
           scale0=1.0, res=0.1, fit_scale=False, max_iter=400):
    """Polish yaw/E/N (and optionally scale) against the DSM.

    Cost blends two complementary terms:
      * chamfer between the mesh footprint boundary and the DSM building
        boundary -- this is what actually pins yaw and translation, because a
        flat roof gives the height term almost nothing to bite on;
      * height RMSE over the overlap -- keeps the superstructures registered.
    """
    from scipy.ndimage import binary_erosion, distance_transform_edt
    from scipy.optimize import minimize

    building = target["building"]
    ys, xs = np.nonzero(building)
    r0, r1 = ys.min() - 40, ys.max() + 41
    c0, c1 = xs.min() - 40, xs.max() + 41
    r0, c0 = max(r0, 0), max(c0, 0)
    r1 = min(r1, building.shape[0])
    c1 = min(c1, building.shape[1])
    sub = building[r0:r1, c0:c1]
    subD = dsm.data.astype(float)[r0:r1, c0:c1]

    edge_t = sub & ~binary_erosion(sub, np.ones((3, 3), bool))
    dist_to_target_edge = distance_transform_edt(~edge_t) * res
    dist_outside = distance_transform_edt(~sub) * res

    gt = dsm.transform
    oe = gt[0] + c0 * gt[1]
    on = gt[3] + r0 * gt[5]
    H, W = sub.shape

    e0, n0, up = _pre_enu(mesh_pts)
    ce, cn = e0.mean(), n0.mean()
    e0, n0 = e0 - ce, n0 - cn
    up_rel = up - deck_up
    # only upward-facing points carry usable height information
    top = mesh_normals_pt[:, 1] > 0.85

    def rasterise(yaw, tE, tN, sc):
        t = np.radians(yaw)
        c, s = np.cos(t), np.sin(t)
        e = sc * (c * e0 - s * n0) + tE
        n = sc * (s * e0 + c * n0) + tN
        z = sc * up_rel
        gi = np.floor((on - n) / res).astype(int)
        gj = np.floor((e - oe) / res).astype(int)
        ok = (gi >= 0) & (gi < H) & (gj >= 0) & (gj < W)
        return gi, gj, ok, z, e, n

    def cost(p):
        yaw, tE, tN = p[0], p[1], p[2]
        sc = p[3] if fit_scale else scale0
        gi, gj, ok, z, e, n = rasterise(yaw, tE, tN, sc)
        if ok.sum() < 500:
            return 1e6
        occ = np.zeros((H, W), bool)
        occ[gi[ok], gj[ok]] = True
        edge_s = occ & ~binary_erosion(occ, np.ones((3, 3), bool))
        if edge_s.sum() < 20:
            return 1e6
        # source edge -> target edge, plus a penalty for source spilling outside
        c1_ = float(dist_to_target_edge[edge_s].mean())
        c2_ = float(dist_outside[occ].mean())
        # height agreement on upward-facing points
        okt = ok & top
        if okt.sum() > 200:
            zs = z[okt] + target["roof_median"]
            zd = subD[gi[okt], gj[okt]]
            good = np.isfinite(zd)
            resid = zs[good] - zd[good]
            resid = resid - np.median(resid)
            c3_ = float(np.sqrt(np.mean(np.clip(resid, -3, 3) ** 2)))
        else:
            c3_ = 3.0
        return c1_ + 0.5 * c2_ + 0.3 * c3_

    x0 = [yaw0, tE0, tN0] + ([scale0] if fit_scale else [])
    steps = [1.5, 0.4, 0.4] + ([0.01] if fit_scale else [])
    simplex = [np.array(x0, float)]
    for i in range(len(x0)):
        p = np.array(x0, float)
        p[i] += steps[i]
        simplex.append(p)
    r = minimize(cost, np.array(x0, float), method="Nelder-Mead",
                 options={"initial_simplex": np.array(simplex),
                          "maxiter": max_iter, "xatol": 1e-3, "fatol": 1e-5})

    yaw, tE, tN = r.x[0], r.x[1], r.x[2]
    sc = r.x[3] if fit_scale else scale0

    # Vertical offset from upward-facing points, robust to outliers. Sample the
    # DSM bilinearly (matching verify_alignment) so the two agree.
    gi, gj, ok, z, e, n = rasterise(yaw, tE, tN, sc)
    # Tie the offset to the deck surface specifically. Parapet copings sit
    # ~0.6 m proud of the deck but the DSM renders them at roughly roof level,
    # so including them biases tZ upward by ~0.2 m.
    on_deck = ok & top & (np.abs(up_rel) < 0.25)
    sel = on_deck if on_deck.sum() > 500 else (ok & top)
    zd = dsm.sample(e[sel], n[sel])
    good = np.isfinite(zd)
    resid = zd[good] - z[sel][good]
    dz = float(np.median(resid))

    return {
        "yaw_deg": float(yaw % 360.0), "scale": float(sc),
        "tE": float(tE), "tN": float(tN), "tZ": dz,
        "pivot_pre": [float(ce), float(cn), float(deck_up)],
        "cost": float(r.fun), "cost_start": float(cost(np.array(x0, float))),
        "n_iter": int(r.nit), "success": bool(r.success),
        "deck_up_local": float(deck_up),
    }


def matrix_from_solution(sol, tilt_correction=None):
    return build_matrix(sol["yaw_deg"], sol["scale"],
                        [sol["tE"], sol["tN"], sol["tZ"]],
                        pivot=sol["pivot_pre"], tilt_correction=tilt_correction)


def verify_alignment(mesh, M, target, dsm, res=0.1):
    """Independent QC of a finished transform: footprint IoU + height residuals."""
    from scipy.ndimage import binary_fill_holes

    P, Npt, W, _ = mesh.sample_surface_with_normals(10)
    Q = apply(M, P)
    Nq = apply_normals(M, Npt)

    building = target["building"]
    ys, xs = np.nonzero(building)
    r0, r1 = max(ys.min() - 30, 0), min(ys.max() + 31, building.shape[0])
    c0, c1 = max(xs.min() - 30, 0), min(xs.max() + 31, building.shape[1])
    sub = building[r0:r1, c0:c1]
    gt = dsm.transform
    oe, on = gt[0] + c0 * gt[1], gt[3] + r0 * gt[5]
    H, W_ = sub.shape

    gi = np.floor((on - Q[:, 1]) / res).astype(int)
    gj = np.floor((Q[:, 0] - oe) / res).astype(int)
    ok = (gi >= 0) & (gi < H) & (gj >= 0) & (gj < W_)
    occ = np.zeros((H, W_), bool)
    occ[gi[ok], gj[ok]] = True
    occ = binary_fill_holes(occ)

    inter = float((occ & sub).sum())
    union = float((occ | sub).sum())

    top = Nq[:, 2] > 0.85
    zd = dsm.sample(Q[top, 0], Q[top, 1])
    good = np.isfinite(zd)
    resid = Q[top, 2][good] - zd[good]
    deck_pts = Q[top][good]
    near_deck = np.abs(deck_pts[:, 2] - target["roof_median"]) < 0.6

    return {
        "footprint_iou": inter / max(union, 1.0),
        "footprint_mesh_m2": float(occ.sum() * res * res),
        "footprint_target_m2": float(sub.sum() * res * res),
        "height_resid_median_m": float(np.median(resid)),
        "height_resid_mad_m": float(np.median(np.abs(resid - np.median(resid)))),
        "height_resid_p95_abs_m": float(np.percentile(np.abs(resid), 95)),
        "deck_mean_elev_m": float(deck_pts[near_deck][:, 2].mean()) if near_deck.sum() else float("nan"),
        "dsm_roof_median_m": float(target["roof_median"]),
        "n_top_points": int(good.sum()),
    }


def residual_tilt(mesh_pts_enu, mesh_normals_enu, dsm, target, max_resid=1.0):
    """Compare the mesh deck plane with the DSM roof plane after alignment.

    A mismatch here is either real drainage fall or ARKit gravity error; we
    report it and let the caller decide whether to apply it.
    """
    top = mesh_normals_enu[:, 2] > 0.98
    P = mesh_pts_enu[top]
    if len(P) < 200:
        return None
    zd = dsm.sample(P[:, 0], P[:, 1])
    ok = np.isfinite(zd)
    P = P[ok]
    # Tight band around the modal deck height: parapet copings sit only ~0.6 m
    # higher and will otherwise tilt the fitted plane.
    hist, edges = np.histogram(P[:, 2], bins=np.arange(P[:, 2].min(),
                                                       P[:, 2].max() + 0.1, 0.1))
    mode_z = edges[int(np.argmax(hist))] + 0.05
    P = P[np.abs(P[:, 2] - mode_z) < 0.30]
    if len(P) < 200:
        return None

    def plane(pts):
        A = np.c_[pts[:, 0] - pts[:, 0].mean(), pts[:, 1] - pts[:, 1].mean(),
                  np.ones(len(pts))]
        coef, *_ = np.linalg.lstsq(A, pts[:, 2], rcond=None)
        tilt = np.degrees(np.arctan(np.hypot(coef[0], coef[1])))
        azi = np.degrees(np.arctan2(-coef[0], -coef[1])) % 360
        return coef, float(tilt), float(azi)

    cm, tm, am = plane(P)
    ys, xs = np.nonzero(target["building"] &
                        (np.abs(dsm.data.astype(float) - target["roof_median"]) < 0.8))
    e, n = dsm.xy(ys, xs)
    Q = np.c_[e, n, dsm.data.astype(float)[ys, xs]]
    cd, td, ad = plane(Q)
    dgrad = np.hypot(cm[0] - cd[0], cm[1] - cd[1])
    return {
        "mesh_deck_tilt_deg": tm, "mesh_deck_downslope_grid_az": am,
        "dsm_roof_tilt_deg": td, "dsm_roof_downslope_grid_az": ad,
        "tilt_disagreement_deg": float(np.degrees(np.arctan(dgrad))),
    }


def tilt_matrix(from_normal, to_normal=(0, 0, 1)):
    """Smallest rotation taking `from_normal` onto `to_normal`."""
    a = np.asarray(from_normal, float)
    a = a / np.linalg.norm(a)
    b = np.asarray(to_normal, float)
    b = b / np.linalg.norm(b)
    v = np.cross(a, b)
    s = np.linalg.norm(v)
    if s < 1e-9:
        return np.eye(3)
    c = float(np.dot(a, b))
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx * ((1 - c) / s**2)


# --------------------------------------------------------------------------
# north
# --------------------------------------------------------------------------
def grid_convergence(lon, lat, lon0):
    """gamma such that true_azimuth = grid_azimuth + gamma (degrees)."""
    return float(np.degrees(np.arctan(np.tan(np.radians(lon - lon0)) *
                                      np.sin(np.radians(lat)))))


def utm_zone_center(crs_wkt, fallback=75.0):
    import re
    m = re.search(r'PARAMETER\["Longitude of natural origin",\s*(-?[\d.]+)', crs_wkt)
    if m:
        return float(m.group(1))
    m = re.search(r'"central_meridian",\s*(-?[\d.]+)', crs_wkt)
    return float(m.group(1)) if m else fallback


def to_lonlat(e, n, crs_wkt):
    """UTM -> lon/lat, via pyproj if available else osgeo."""
    try:
        from pyproj import CRS, Transformer
        tr = Transformer.from_crs(CRS.from_wkt(crs_wkt), CRS.from_epsg(4326),
                                  always_xy=True)
        return tr.transform(e, n)
    except ImportError:
        pass
    try:
        from rasterio.warp import transform as rio_transform
        lon, lat = rio_transform(crs_wkt, "EPSG:4326",
                                 list(np.atleast_1d(e)), list(np.atleast_1d(n)))
        return np.asarray(lon), np.asarray(lat)
    except ImportError:
        from osgeo import osr
        src = osr.SpatialReference()
        src.ImportFromWkt(crs_wkt)
        dst = osr.SpatialReference()
        dst.ImportFromEPSG(4326)
        dst.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
        src.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
        ct = osr.CoordinateTransformation(src, dst)
        pts = ct.TransformPoints(list(zip(np.atleast_1d(e), np.atleast_1d(n))))
        arr = np.asarray(pts)
        return arr[:, 0], arr[:, 1]

In [ ]:
%%writefile rooflib/segment.py
"""Segment the fused rooftop mesh into planes and objects.

Design notes, driven by what this mesh actually is:

* The 8 OBJ `o` groups are texture atlases, not parts, and the mesh is 491
  disconnected shells. So neither the file structure nor topological
  connectivity carries any semantics -- everything here is geometric.

* ~23% of the surface is transition/noise triangles whose pitch and azimuth
  histograms are near-uniform. Plane fitting on raw normals leaves ~48% of the
  area unassigned, so normals are smoothed over a spatial kNN first (the mesh
  is fragmented, so topological neighbours are not available).

* Plane finding is done by normal-direction clustering then offset histogram,
  not brute-force RANSAC: it is far cheaper on 89k faces and buildings are
  overwhelmingly made of a handful of normal directions.

* Naive "cluster everything above the deck" returns ONE blob covering the whole
  roof, because the continuous parapet ring bridges every object. Parapets are
  therefore identified and removed before objects are clustered.
"""
from __future__ import annotations

import numpy as np

HORIZONTAL_MAX_TILT = 12.0   # deg from horizontal to count as a floor/roof
VERTICAL_MIN_TILT = 72.0     # deg from horizontal to count as a wall


# --------------------------------------------------------------------------
# normal smoothing
# --------------------------------------------------------------------------
def smooth_normals(centroids, normals, areas, k=24, iterations=3, radius=0.30,
                   normal_sigma_deg=25.0):
    """Edge-preserving (bilateral) normal smoothing.

    Plain isotropic averaging is actively harmful on a rooftop: at every
    deck/parapet corner it blends a horizontal and a vertical normal into a
    spurious 45 deg one, and measurably *grows* the noise band. Neighbours are
    therefore weighted by normal similarity as well as distance, so smoothing
    happens along surfaces and stops at creases.
    """
    from scipy.spatial import cKDTree

    tree = cKDTree(centroids)
    nbrs = tree.query_ball_point(centroids, r=radius)
    cos_sigma = np.radians(normal_sigma_deg)
    N = normals.copy()
    for _ in range(iterations):
        out = np.empty_like(N)
        for i, nb in enumerate(nbrs):
            if len(nb) < 3:
                out[i] = N[i]
                continue
            nb = np.asarray(nb)
            if len(nb) > k:
                d2 = ((centroids[nb] - centroids[i]) ** 2).sum(1)
                nb = nb[np.argsort(d2)[:k]]
            ni = N[nb]
            sgn = np.sign(ni @ N[i])
            sgn[sgn == 0] = 1
            ni = sgn[:, None] * ni
            ang = np.arccos(np.clip(ni @ N[i], -1, 1))
            w = areas[nb] * np.exp(-0.5 * (ang / cos_sigma) ** 2)
            v = (w[:, None] * ni).sum(0)
            nv = np.linalg.norm(v)
            out[i] = v / nv if nv > 1e-9 else N[i]
        N = out
    return N


def assign_residual_faces(planes, centroids, normals, areas, assigned,
                          dist_tol=0.10, angle_tol_deg=18.0, rounds=3):
    """Sweep leftover faces into the nearest compatible plane.

    Direction-cluster + offset-peak finding is conservative by construction and
    leaves the crease geometry unclaimed; this recovers most of it without
    inventing new planes.
    """
    cos_tol = np.cos(np.radians(angle_tol_deg))
    normals_p = np.array([p["normal"] for p in planes])
    offsets_p = np.array([p["normal"] @ p["centroid"] for p in planes])
    for _ in range(rounds):
        free = np.nonzero(~assigned)[0]
        if len(free) == 0:
            break
        ang = np.abs(normals[free] @ normals_p.T)
        dist = np.abs(centroids[free] @ normals_p.T - offsets_p[None, :])
        ok = (ang >= cos_tol) & (dist <= dist_tol)
        score = np.where(ok, -dist, -np.inf)
        best = np.argmax(score, axis=1)
        good = np.isfinite(score[np.arange(len(free)), best])
        if not good.any():
            break
        for pid in np.unique(best[good]):
            sel = free[good & (best == pid)]
            planes[pid]["faces"] = np.concatenate([planes[pid]["faces"], sel])
            assigned[sel] = True
        # refit after absorbing new members
        for p in planes:
            w = areas[p["faces"]]
            nrm = (w[:, None] * normals[p["faces"]]).sum(0)
            n_ = np.linalg.norm(nrm)
            if n_ > 1e-9:
                p["normal"] = nrm / n_
            p["centroid"] = (w[:, None] * centroids[p["faces"]]).sum(0) / w.sum()
            p["area_m2"] = float(w.sum())
        normals_p = np.array([p["normal"] for p in planes])
        offsets_p = np.array([p["normal"] @ p["centroid"] for p in planes])
    return planes, assigned


# --------------------------------------------------------------------------
# plane extraction
# --------------------------------------------------------------------------
def _direction_clusters(normals, areas, angle_tol_deg=12.0, min_area=1.0):
    """Greedy clustering of normal directions by area."""
    cos_tol = np.cos(np.radians(angle_tol_deg))
    remaining = np.ones(len(normals), bool)
    clusters = []
    while remaining.any():
        idx = np.nonzero(remaining)[0]
        # seed on the largest-area remaining face's direction
        seed = idx[np.argmax(areas[idx])]
        d = normals[seed].copy()
        for _ in range(6):  # a few refits to centre the cluster
            sel = idx[(normals[idx] @ d) >= cos_tol]
            if len(sel) == 0:
                break
            v = (areas[sel][:, None] * normals[sel]).sum(0)
            nv = np.linalg.norm(v)
            if nv < 1e-9:
                break
            d = v / nv
        sel = idx[(normals[idx] @ d) >= cos_tol]
        if len(sel) == 0:
            remaining[seed] = False
            continue
        if areas[sel].sum() >= min_area:
            clusters.append((d, sel))
        remaining[sel] = False
    return clusters


def _split_by_offset(centroids, areas, direction, members, bin_size=0.12,
                     min_area=0.8):
    """Split a normal-direction cluster into parallel planes by offset peaks."""
    off = centroids[members] @ direction
    lo, hi = off.min(), off.max()
    if hi - lo < bin_size:
        return [(float(off.mean()), members)]
    edges = np.arange(lo, hi + bin_size, bin_size)
    hist, _ = np.histogram(off, bins=edges, weights=areas[members])
    peaks = []
    for i in range(len(hist)):
        if hist[i] <= 0:
            continue
        left = hist[i - 1] if i > 0 else 0.0
        right = hist[i + 1] if i + 1 < len(hist) else 0.0
        if hist[i] >= left and hist[i] >= right and hist[i] >= min_area * 0.5:
            peaks.append(i)
    if not peaks:
        peaks = [int(np.argmax(hist))]
    centers = np.array([edges[i] + bin_size / 2 for i in peaks])
    assign = np.argmin(np.abs(off[:, None] - centers[None, :]), axis=1)
    out = []
    for j in range(len(centers)):
        sel = members[assign == j]
        if len(sel) and areas[sel].sum() >= min_area:
            out.append((float(np.average(off[assign == j],
                                         weights=areas[sel])), sel))
    return out


def _spatial_split(centroids, areas, direction, offset, members, cell=0.3,
                   min_area=0.6):
    """Break a plane into spatially disconnected patches (two aligned walls etc.)."""
    from scipy.ndimage import label

    d = np.asarray(direction, float)
    tmp = np.array([0.0, 0.0, 1.0]) if abs(d[2]) < 0.9 else np.array([1.0, 0.0, 0.0])
    u = np.cross(d, tmp)
    u /= np.linalg.norm(u)
    v = np.cross(d, u)
    P = centroids[members]
    a = P @ u
    b = P @ v
    ai = np.floor((a - a.min()) / cell).astype(int)
    bi = np.floor((b - b.min()) / cell).astype(int)
    H, W = ai.max() + 3, bi.max() + 3
    grid = np.zeros((H, W), bool)
    grid[ai + 1, bi + 1] = True
    lab, n = label(grid, structure=np.ones((3, 3), int))
    ids = lab[ai + 1, bi + 1]
    out = []
    for c in range(1, n + 1):
        sel = members[ids == c]
        if len(sel) and areas[sel].sum() >= min_area:
            out.append(sel)
    return out


def extract_planes(centroids, normals, areas, angle_tol_deg=12.0,
                   offset_bin=0.12, min_area=0.8, spatial_cell=0.3):
    """Full plane decomposition. Returns a list of dicts."""
    planes = []
    for direction, members in _direction_clusters(normals, areas,
                                                  angle_tol_deg, min_area):
        for offset, sub in _split_by_offset(centroids, areas, direction,
                                            members, offset_bin, min_area):
            for patch in _spatial_split(centroids, areas, direction, offset,
                                        sub, spatial_cell, min_area * 0.75):
                w = areas[patch]
                nrm = (w[:, None] * normals[patch]).sum(0)
                nrm /= np.linalg.norm(nrm)
                if nrm[2] < 0:          # keep normals pointing up / outward-up
                    pass                 # (walls keep their own orientation)
                cen = (w[:, None] * centroids[patch]).sum(0) / w.sum()
                planes.append({
                    "faces": patch,
                    "normal": nrm,
                    "centroid": cen,
                    "area_m2": float(w.sum()),
                })
    planes.sort(key=lambda p: -p["area_m2"])
    for i, p in enumerate(planes):
        p["plane_id"] = i
    return planes


def plane_geometry(plane, convergence_deg=0.0):
    """Tilt, azimuth (grid and true), and projected area for one plane."""
    n = plane["normal"]
    nz = n if n[2] >= 0 else -n          # upward-facing convention for tilt/azimuth
    tilt = float(np.degrees(np.arccos(np.clip(nz[2], -1, 1))))
    # Below ~2 deg the in-plane direction is numerically meaningless -- a "flat"
    # roof has no aspect, and reporting one invites it being used as a mounting
    # azimuth downstream.
    if tilt < 2.0:
        grid_az = float("nan")
    else:
        grid_az = float(np.degrees(np.arctan2(nz[0], nz[1])) % 360.0)
    horiz = tilt <= HORIZONTAL_MAX_TILT
    vert = tilt >= VERTICAL_MIN_TILT
    return {
        "tilt_deg": tilt,
        "grid_azimuth_deg": grid_az,
        "true_azimuth_deg": (grid_az + convergence_deg) % 360.0 if np.isfinite(grid_az) else float("nan"),
        "projected_area_m2": float(plane["area_m2"] * abs(nz[2])),
        "orientation": "horizontal" if horiz else ("vertical" if vert else "sloped"),
    }


# --------------------------------------------------------------------------
# object isolation
# --------------------------------------------------------------------------
def height_grid(points, weights_z, cell=0.1, pad=4):
    """Top-down max-height raster of a point set; returns (grid, origin, cell)."""
    e, n = points[:, 0], points[:, 1]
    e0, n1 = e.min() - pad * cell, n.max() + pad * cell
    gj = np.floor((e - e0) / cell).astype(int)
    gi = np.floor((n1 - n) / cell).astype(int)
    H = gi.max() + pad + 1
    W = gj.max() + pad + 1
    g = np.full((H, W), -1e9)
    np.maximum.at(g, (gi, gj), weights_z)
    return np.where(g > -1e8, g, np.nan), (e0, n1), cell


def footprint_mask(points_xy, cell=0.1, close_iter=3):
    """Solid top-down footprint of a point set, plus its raster origin."""
    from scipy.ndimage import binary_closing, binary_fill_holes

    e0 = points_xy[:, 0].min() - 5 * cell
    n1 = points_xy[:, 1].max() + 5 * cell
    gj = np.floor((points_xy[:, 0] - e0) / cell).astype(int)
    gi = np.floor((n1 - points_xy[:, 1]) / cell).astype(int)
    H, W = gi.max() + 6, gj.max() + 6
    occ = np.zeros((H, W), bool)
    occ[gi, gj] = True
    occ = binary_closing(occ, np.ones((2 * close_iter + 1,) * 2, bool))
    occ = binary_fill_holes(occ)
    return occ, (e0, n1), cell


def boundary_distance_fn(mask, origin, cell):
    """Callable: (E, N) -> metres from the footprint boundary (0 on the edge)."""
    from scipy.ndimage import binary_erosion, distance_transform_edt

    edge = mask & ~binary_erosion(mask, np.ones((3, 3), bool))
    dist = distance_transform_edt(~edge) * cell
    e0, n1 = origin
    H, W = mask.shape

    def fn(xy):
        xy = np.atleast_2d(np.asarray(xy, float))
        gj = np.clip(np.floor((xy[:, 0] - e0) / cell).astype(int), 0, W - 1)
        gi = np.clip(np.floor((n1 - xy[:, 1]) / cell).astype(int), 0, H - 1)
        out = dist[gi, gj]
        return out[0] if out.size == 1 else out

    return fn


def find_parapet(planes, geoms, footprint_poly_dist, deck_z,
                 max_top_above_deck=1.8, max_dist_from_edge=1.2):
    """Vertical planes hugging the roof outline and short enough to be a parapet."""
    ids = []
    for p, g in zip(planes, geoms):
        if g["orientation"] != "vertical":
            continue
        top = p["z_max"]
        if top - deck_z > max_top_above_deck:
            continue
        if footprint_poly_dist(p["centroid"][:2]) > max_dist_from_edge:
            continue
        ids.append(p["plane_id"])
    return set(ids)


def cluster_objects(face_centroids, face_areas, face_z, deck_z, exclude_faces,
                    min_height=0.35, cell=0.15, min_area=0.4, gap_cells=1):
    """Connected-component clustering of above-deck geometry.

    `exclude_faces` must already remove the parapet ring, otherwise everything
    merges into a single component.
    """
    from scipy.ndimage import binary_closing, label

    keep = (face_z - deck_z > min_height) & (~exclude_faces)
    idx = np.nonzero(keep)[0]
    if len(idx) == 0:
        return [], None
    P = face_centroids[idx]
    e0, n1 = P[:, 0].min() - cell, P[:, 1].max() + cell
    gj = np.floor((P[:, 0] - e0) / cell).astype(int)
    gi = np.floor((n1 - P[:, 1]) / cell).astype(int)
    H, W = gi.max() + 2, gj.max() + 2
    occ = np.zeros((H, W), bool)
    occ[gi, gj] = True
    if gap_cells:
        occ = binary_closing(occ, np.ones((2 * gap_cells + 1,) * 2, bool))
    lab, n = label(occ, structure=np.ones((3, 3), int))
    ids = lab[gi, gj]
    out = []
    for c in range(1, n + 1):
        sel = idx[ids == c]
        if len(sel) == 0 or face_areas[sel].sum() < min_area:
            continue
        out.append(sel)
    out.sort(key=lambda s: -face_areas[s].sum())
    return out, (lab, (e0, n1), cell)


def extract_objects_hierarchical(planes, geoms, centroids, areas, normals,
                                 min_terrace_area=3.0, min_height=0.35,
                                 cell=0.15, min_object_area=0.4,
                                 footprint_cell=0.15, dilate_m=0.4):
    """Peel objects off the roof one terrace at a time, highest terrace first.

    A flat "everything above the deck" clustering fails on this building twice
    over: the parapet ring bridges separate objects, and the roof is genuinely
    layered (open deck -> a ~3 m block -> water tanks standing on that block),
    so a single threshold cannot separate the tanks from the block carrying
    them.

    Working top-down fixes both. Each horizontal plane is a terrace; objects
    resting on it are the above-terrace geometry inside its footprint that no
    higher terrace has already claimed. The result is a parent/child tree
    rather than a flat list.
    """
    from scipy.ndimage import binary_dilation, binary_closing, label

    terraces = [(p, g) for p, g in zip(planes, geoms)
                if g["orientation"] == "horizontal" and p["area_m2"] >= min_terrace_area]
    terraces.sort(key=lambda pg: -pg[0]["centroid"][2])  # highest first

    used = np.zeros(len(areas), bool)
    for p, _ in terraces:
        used[p["faces"]] = True   # terrace surfaces are not objects themselves

    results = []
    for p, g in terraces:
        z_t = float(p["centroid"][2])
        fmask, origin, fcell = footprint_mask(centroids[p["faces"]][:, :2],
                                              cell=footprint_cell)
        grow = int(round(dilate_m / fcell))
        if grow > 0:
            fmask = binary_dilation(fmask, np.ones((2 * grow + 1,) * 2, bool))
        e0, n1 = origin
        H, W = fmask.shape

        gj = np.floor((centroids[:, 0] - e0) / fcell).astype(int)
        gi = np.floor((n1 - centroids[:, 1]) / fcell).astype(int)
        inside = ((gi >= 0) & (gi < H) & (gj >= 0) & (gj < W))
        over = np.zeros(len(areas), bool)
        over[inside] = fmask[gi[inside], gj[inside]]

        cand = over & (~used) & (centroids[:, 2] > z_t + min_height)
        idx = np.nonzero(cand)[0]
        children = []
        if len(idx):
            # Cluster in 3D voxels, not in top-down projection: two tanks a
            # hand's width apart share footprint cells once projected, and a
            # ring of walls encloses interior cells that merge everything into
            # one component. In 3D they stay distinct.
            P = centroids[idx]
            o = P.min(0) - cell
            vi = np.floor((P - o) / cell).astype(int)
            dims = vi.max(0) + 2
            occ = np.zeros(dims, bool)
            occ[vi[:, 0], vi[:, 1], vi[:, 2]] = True
            lab, n = label(occ, structure=np.ones((3, 3, 3), int))
            ids = lab[vi[:, 0], vi[:, 1], vi[:, 2]]
            for c in range(1, n + 1):
                sel = idx[ids == c]
                if len(sel) and areas[sel].sum() >= min_object_area:
                    children.append(sel)
            children.sort(key=lambda s: -areas[s].sum())
            for s in children:
                used[s] = True

        results.append({
            "terrace_plane_id": p["plane_id"],
            "terrace_z": z_t,
            "terrace_area_m2": p["area_m2"],
            "objects": children,
        })

    results.sort(key=lambda r: r["terrace_z"])
    return results


def classify_object(verts_enu, faces_sel, centroids, normals, areas, deck_z):
    """Heuristic label for an above-deck cluster."""
    C = centroids[faces_sel]
    A = areas[faces_sel]
    N = normals[faces_sel]
    z = C[:, 2]
    height = float(z.max() - deck_z)
    base = float(np.percentile(z, 2) - deck_z)
    # np.ptp(), not ndarray.ptp() -- the method was removed in numpy 2.0
    foot_e = float(np.ptp(C[:, 0]))
    foot_n = float(np.ptp(C[:, 1]))
    foot = max(foot_e * foot_n, 1e-6)
    tilt = np.degrees(np.arccos(np.clip(np.abs(N[:, 2]), 0, 1)))
    horiz_area = float(A[tilt < HORIZONTAL_MAX_TILT].sum())
    vert_area = float(A[tilt > VERTICAL_MIN_TILT].sum())
    total = float(A.sum())

    # rectilinearity: how concentrated the wall azimuths are into 2 orthogonal
    # families. Tanks are round, so theirs spread out.
    wall = tilt > VERTICAL_MIN_TILT
    if wall.sum() > 10:
        az = np.degrees(np.arctan2(N[wall, 0], N[wall, 1])) % 180.0
        h, _ = np.histogram(az, bins=18, range=(0, 180), weights=A[wall])
        rect = float(np.sort(h)[-2:].sum() / max(h.sum(), 1e-9))
    else:
        rect = 0.0

    # Labels are heuristic and deliberately conservative. On a noisy phone-LiDAR
    # scan the geometric signature of a stair headroom and a walled tank plinth
    # overlap heavily, so anything ambiguous stays generic and carries a low
    # confidence rather than asserting a type that downstream code would trust.
    label, confidence = "unclassified_object", 0.3
    if foot > 20.0 and height > 1.5:
        label, confidence = "superstructure_block", 0.5
    elif height > 1.0 and foot <= 6.0 and rect < 0.60:
        label, confidence = "water_tank", 0.6 if height > 1.5 else 0.45
    elif 4.0 < foot <= 20.0 and 1.6 < height < 3.4 and rect >= 0.60 and horiz_area > 0.6:
        label, confidence = "stair_headroom", 0.55
    elif height <= 1.0 and horiz_area > 0.5 * total:
        label, confidence = "raised_platform", 0.5
    elif total < 1.5:
        label, confidence = "small_object", 0.4
    if base > 0.8:
        label = label + "_elevated"
    return {
        "label": label,
        "label_confidence": confidence,
        "height_above_deck_m": height,
        "base_above_deck_m": base,
        "top_elev_m": float(z.max()),
        "base_elev_m": float(np.percentile(z, 2)),
        "footprint_bbox_m": [foot_e, foot_n],
        "surface_area_m2": total,
        "horizontal_area_m2": horiz_area,
        "vertical_area_m2": vert_area,
        "rectilinearity": rect,
        "centroid_e": float(C[:, 0].mean()),
        "centroid_n": float(C[:, 1].mean()),
    }

In [ ]:
%%writefile rooflib/solar.py
"""Solar-facing attributes: aspect, usable area, obstruction context, shading.

Two conventions matter and are easy to get wrong:

* Azimuths derived from a UTM grid are GRID azimuths. Solar geometry needs TRUE
  north. At this site grid convergence is -0.951 deg, so true = grid - 0.951.
  Every azimuth leaving this module is true-north referenced and labelled.

* A roof flatter than ~2 deg has no meaningful aspect. Reporting one anyway
  invites a downstream layout tool mounting panels to a number that is pure
  fitting noise, so it is returned as NaN.
"""
from __future__ import annotations

import numpy as np


def plane_solar_record(plane, geom, centroids, areas, deck_z, convergence_deg):
    """Per-plane record with everything a layout/yield step needs."""
    F = plane["faces"]
    C = centroids[F]
    z = C[:, 2]
    tilt = geom["tilt_deg"]
    flat = tilt < 2.0
    return {
        "plane_id": int(plane["plane_id"]),
        "orientation": geom["orientation"],
        "surface_area_m2": round(float(plane["area_m2"]), 3),
        "projected_area_m2": round(float(geom["projected_area_m2"]), 3),
        "tilt_deg": round(tilt, 3),
        "grid_azimuth_deg": None if flat else round(geom["grid_azimuth_deg"], 2),
        "true_azimuth_deg": None if flat else round(geom["true_azimuth_deg"], 2),
        "aspect": "flat" if flat else _compass(geom["true_azimuth_deg"]),
        "centroid_e": round(float(plane["centroid"][0]), 3),
        "centroid_n": round(float(plane["centroid"][1]), 3),
        "centroid_z": round(float(plane["centroid"][2]), 3),
        "z_min": round(float(z.min()), 3),
        "z_max": round(float(z.max()), 3),
        "height_above_deck_m": round(float(plane["centroid"][2] - deck_z), 3),
        "n_faces": int(len(F)),
    }


def _compass(az):
    if not np.isfinite(az):
        return "flat"
    names = ["N", "NNE", "NE", "ENE", "E", "ESE", "SE", "SSE",
             "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW"]
    return names[int(((az % 360) + 11.25) // 22.5) % 16]


# --------------------------------------------------------------------------
# usable area on a terrace
# --------------------------------------------------------------------------
def usable_area(terrace_plane, centroids, obstruction_face_sets, cell=0.1,
                edge_setback_m=0.6, obstruction_clearance_m=0.5):
    """Deck area left after edge setback and clearance around obstructions.

    This is the number a layout step actually consumes -- gross plane area
    overstates what panels can occupy, often by 30-40 % on a small roof.
    """
    from scipy.ndimage import binary_dilation, binary_erosion

    from .segment import footprint_mask

    mask, origin, c = footprint_mask(centroids[terrace_plane["faces"]][:, :2],
                                     cell=cell)
    gross = float(mask.sum() * c * c)

    er = int(round(edge_setback_m / c))
    inner = binary_erosion(mask, np.ones((2 * er + 1,) * 2, bool)) if er > 0 else mask

    e0, n1 = origin
    H, W = mask.shape
    blocked = np.zeros_like(mask)
    for sel in obstruction_face_sets:
        P = centroids[sel]
        gj = np.floor((P[:, 0] - e0) / c).astype(int)
        gi = np.floor((n1 - P[:, 1]) / c).astype(int)
        ok = (gi >= 0) & (gi < H) & (gj >= 0) & (gj < W)
        blocked[gi[ok], gj[ok]] = True
    cl = int(round(obstruction_clearance_m / c))
    if cl > 0 and blocked.any():
        blocked = binary_dilation(blocked, np.ones((2 * cl + 1,) * 2, bool))

    free = inner & ~blocked
    return {
        "gross_area_m2": round(gross, 2),
        "after_setback_m2": round(float(inner.sum() * c * c), 2),
        "usable_area_m2": round(float(free.sum() * c * c), 2),
        "blocked_by_objects_m2": round(float((inner & blocked).sum() * c * c), 2),
        "edge_setback_m": edge_setback_m,
        "obstruction_clearance_m": obstruction_clearance_m,
        "_mask": free,
        "_origin": origin,
        "_cell": c,
    }


# --------------------------------------------------------------------------
# shading
# --------------------------------------------------------------------------
def horizon_profile(dsm, origin_e, origin_n, origin_z, n_sectors=36,
                    max_dist_m=60.0, step_m=0.5, convergence_deg=0.0):
    """Sky-obstruction elevation angle per azimuth sector, from the DSM.

    The DSM is the only input that sees beyond the building, so it is the only
    source for neighbour shading. Returned azimuths are true-north referenced.
    """
    out = []
    dists = np.arange(2.0, max_dist_m + step_m, step_m)
    for k in range(n_sectors):
        grid_az = 360.0 * k / n_sectors
        th = np.radians(grid_az)
        e = origin_e + dists * np.sin(th)
        n = origin_n + dists * np.cos(th)
        z = dsm.sample(e, n)
        good = np.isfinite(z)
        if not good.any():
            out.append({"true_azimuth_deg": (grid_az + convergence_deg) % 360,
                        "horizon_elev_deg": 0.0, "distance_m": None,
                        "obstruction_height_m": 0.0})
            continue
        ang = np.degrees(np.arctan2(z[good] - origin_z, dists[good]))
        i = int(np.argmax(ang))
        out.append({
            "true_azimuth_deg": round((grid_az + convergence_deg) % 360, 2),
            "horizon_elev_deg": round(float(max(ang[i], 0.0)), 3),
            "distance_m": round(float(dists[good][i]), 2),
            "obstruction_height_m": round(float(z[good][i] - origin_z), 3),
        })
    return out


def sector_summary(dsm, mask, target, deck_z, n_sectors=12, max_dist_m=50.0,
                   min_dist_m=8.0, convergence_deg=0.0):
    """Coarse per-sector obstruction table around the building."""
    D = dsm.data.astype(float)
    ys, xs = np.nonzero(mask.data == 1)
    cy, cx = ys.mean(), xs.mean()
    Y, X = np.mgrid[0:D.shape[0], 0:D.shape[1]]
    res = dsm.res[0]
    dist = np.hypot((Y - cy) * res, (X - cx) * res)
    grid_bearing = np.degrees(np.arctan2((X - cx) * res, -(Y - cy) * res)) % 360
    outside = mask.data != 1
    width = 360.0 / n_sectors
    rows = []
    for k in range(n_sectors):
        a0 = k * width
        sel = outside & (grid_bearing >= a0) & (grid_bearing < a0 + width) \
            & (dist < max_dist_m) & (dist > min_dist_m)
        if sel.sum() == 0:
            continue
        zz = D[sel]
        i = int(np.argmax(zz))
        rows.append({
            "sector_true_az_from": round((a0 + convergence_deg) % 360, 2),
            "sector_true_az_to": round((a0 + width + convergence_deg) % 360, 2),
            "max_elev_m": round(float(zz.max()), 2),
            "above_deck_m": round(float(zz.max() - deck_z), 2),
            "distance_at_max_m": round(float(dist[sel][i]), 1),
            "elev_angle_deg": round(float(np.degrees(np.arctan2(
                max(zz.max() - deck_z, 0.0), max(dist[sel][i], 1e-6)))), 2),
            "px_above_deck": int((zz > deck_z).sum()),
        })
    return rows


def solar_position_summary(lat, lon):
    """Noon sun elevation at solstices/equinox -- a sanity frame for shading.

    Deliberately a closed-form declination model, not an ephemeris: it is used
    to sanity-check that computed horizon angles matter, not to drive yield.
    """
    out = {}
    for name, decl in (("summer_solstice", 23.44), ("equinox", 0.0),
                       ("winter_solstice", -23.44)):
        elev = 90.0 - abs(lat - decl)
        out[name] = {
            "solar_declination_deg": decl,
            "solar_noon_elevation_deg": round(elev, 2),
            "shadow_length_per_m": round(1.0 / np.tan(np.radians(elev)), 3)
            if elev > 1 else None,
        }
    out["latitude_deg"] = round(lat, 6)
    out["longitude_deg"] = round(lon, 6)
    out["hemisphere"] = "northern" if lat >= 0 else "southern"
    out["optimal_fixed_tilt_deg_rule_of_thumb"] = round(abs(lat) * 0.87, 1)
    out["optimal_azimuth_true_deg"] = 180.0 if lat >= 0 else 0.0
    return out


def interrow_spacing(obstruction_height_m, min_sun_elev_deg=25.0):
    """Shadow length cast by an obstruction at a given sun elevation."""
    if obstruction_height_m <= 0:
        return 0.0
    return float(obstruction_height_m / np.tan(np.radians(min_sun_elev_deg)))

In [ ]:
%%writefile rooflib/exporters.py
"""Export the segmented, georeferenced model.

Two coordinate flavours are written, as requested:

* local ENU -- origin at the building centroid, metres. Vertex values stay
  small, which matters because most 3D viewers store positions as float32:
  at UTM easting 250811 / northing 2546703 the representable step is already
  ~0.25 m, so an absolute-coordinate mesh visibly jitters and z-fights.
* absolute UTM 43N -- for GIS interop, where that precision loss does not
  apply because GIS stores doubles.

`origin_utm` in the sidecar JSON converts one to the other.
"""
from __future__ import annotations

import json

import numpy as np


def write_obj(path, verts, faces, groups=None, group_names=None,
              header_comment=None):
    """Write an OBJ, optionally splitting faces into named `o` groups."""
    with open(path, "w") as fh:
        if header_comment:
            for line in header_comment.splitlines():
                fh.write(f"# {line}\n")
        for v in verts:
            fh.write("v %.6f %.6f %.6f\n" % (v[0], v[1], v[2]))
        if groups is None:
            for f in faces:
                fh.write("f %d %d %d\n" % (f[0] + 1, f[1] + 1, f[2] + 1))
        else:
            groups = np.asarray(groups)
            for gid in np.unique(groups):
                name = (group_names or {}).get(int(gid), f"segment_{int(gid)}")
                fh.write(f"o {name}\n")
                for f in faces[groups == gid]:
                    fh.write("f %d %d %d\n" % (f[0] + 1, f[1] + 1, f[2] + 1))


def write_ply_labelled(path, verts, faces, face_labels, palette=None):
    """ASCII PLY with a per-face colour -- opens in MeshLab/CloudCompare."""
    labels = np.asarray(face_labels)
    uniq = np.unique(labels)
    if palette is None:
        rng = np.random.default_rng(7)
        palette = {int(u): tuple(int(x) for x in rng.integers(60, 235, 3))
                   for u in uniq}
    with open(path, "w") as fh:
        fh.write("ply\nformat ascii 1.0\n")
        fh.write(f"element vertex {len(verts)}\n")
        fh.write("property float x\nproperty float y\nproperty float z\n")
        fh.write(f"element face {len(faces)}\n")
        fh.write("property list uchar int vertex_index\n")
        fh.write("property uchar red\nproperty uchar green\nproperty uchar blue\n")
        fh.write("end_header\n")
        for v in verts:
            fh.write("%.6f %.6f %.6f\n" % (v[0], v[1], v[2]))
        for f, l in zip(faces, labels):
            r, g, b = palette[int(l)]
            fh.write("3 %d %d %d %d %d %d\n" % (f[0], f[1], f[2], r, g, b))


def write_geojson(path, features, crs_epsg=32643):
    fc = {
        "type": "FeatureCollection",
        "name": "roof_segments",
        "crs": {"type": "name",
                "properties": {"name": f"urn:ogc:def:crs:EPSG::{crs_epsg}"}},
        "features": features,
    }
    with open(path, "w") as fh:
        json.dump(fc, fh, indent=2)


def mask_to_polygons(mask, origin, cell, min_area_cells=4):
    """Trace raster components into (already simplified) rectilinear rings.

    Marching-squares would give smoother outlines but pulls in extra
    dependencies; per-component boundary tracing is enough for a footprint
    polygon at 0.1 m and keeps the export dependency-free.
    """
    from scipy.ndimage import label

    e0, n1 = origin
    lab, n = label(mask, structure=np.ones((3, 3), int))
    polys = []
    for c in range(1, n + 1):
        sel = lab == c
        if sel.sum() < min_area_cells:
            continue
        ii, jj = np.nonzero(sel)
        ring = []
        # boundary cells only, ordered by angle about the centroid: adequate
        # for convex-ish roof parts, which is what these are
        from scipy.ndimage import binary_erosion
        edge = sel & ~binary_erosion(sel, np.ones((3, 3), bool))
        ei, ej = np.nonzero(edge)
        ex = e0 + (ej + 0.5) * cell
        ny = n1 - (ei + 0.5) * cell
        cx, cy = ex.mean(), ny.mean()
        order = np.argsort(np.arctan2(ny - cy, ex - cx))
        ring = [[float(ex[k]), float(ny[k])] for k in order]
        if len(ring) < 3:
            continue
        ring.append(ring[0])
        polys.append({"ring": ring, "area_m2": float(sel.sum() * cell * cell)})
    return polys


def transform_sidecar(solution, matrix, origin_utm, crs_wkt, convergence_deg,
                      diagnostics, notes=None):
    """Everything needed to reproduce or invert the georeferencing."""
    return {
        "schema": "rooflib/transform/1",
        "description": "ARKit mesh -> georeferenced ENU / UTM",
        "source_frame": {
            "name": "ARKit world (gravity-aligned)",
            "axes": "x right, y up (anti-gravity), z toward viewer",
            "units": "metres",
        },
        "axis_map_arkit_to_enu": [[1, 0, 0], [0, 0, -1], [0, 1, 0]],
        "axis_map_note": "east=+x, north=-z, up=+y; determinant +1 "
                         "(chirality preserved). Using north=+z mirrors the "
                         "model and flips every azimuth.",
        "solution": {
            "yaw_deg_grid": solution["yaw_deg"],
            "scale": solution["scale"],
            "translation_e": solution["tE"],
            "translation_n": solution["tN"],
            "translation_z": solution["tZ"],
            "pivot_pre_enu": solution["pivot_pre"],
        },
        "matrix_arkit_to_utm_4x4": np.asarray(matrix).tolist(),
        "local_enu": {
            "origin_utm": list(map(float, origin_utm)),
            "note": "local = utm - origin_utm; keeps float32 viewers precise",
        },
        "crs": {
            "horizontal_wkt": crs_wkt,
            "horizontal_epsg": 32643,
            "vertical_datum": "UNDECLARED in source DSM -- elevations are "
                              "carried through as-is from dsm.tif. Could be "
                              "ellipsoidal or a geoid/MSL height. Does not "
                              "affect tilt, azimuth or yield.",
        },
        "north": {
            "grid_convergence_deg": convergence_deg,
            "relation": "true_azimuth = grid_azimuth + grid_convergence_deg",
            "note": "all exported azimuths are TRUE north referenced",
        },
        "diagnostics": diagnostics,
        "notes": notes or [],
    }


def save_json(path, obj):
    def default(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, (np.bool_,)):
            return bool(o)
        raise TypeError(f"{type(o)} not serialisable")

    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, default=default)


def write_csv(path, rows, columns=None):
    import csv
    if not rows:
        open(path, "w").close()
        return
    columns = columns or list(rows[0].keys())
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

In [ ]:
%%writefile rooflib/viz.py
"""QC figures.

Palette is the validated 3-slot categorical set (blue/orange/aqua); it clears
the all-pairs CVD and normal-vision floors in light mode, which matters here
because the segmentation map is a spatial plot where any pair of classes can
end up adjacent. Aqua sits below 3:1 against the surface, so every class is
also carried by a visible legend label -- identity is never colour-alone.
"""
from __future__ import annotations

import numpy as np

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#8a8a86"
GRID = "#e3e3df"

# categorical slots, fixed order, never cycled
C1 = "#2a78d6"   # blue
C2 = "#eb6834"   # orange
C3 = "#1baf7a"   # aqua
C4 = "#eda100"   # yellow
C8 = "#e34948"   # red

CLASS_COLORS = {"horizontal": C1, "vertical": C2, "sloped": C3}


def _style(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK_2, labelsize=8, length=3)
    ax.grid(True, color=GRID, linewidth=0.6, alpha=0.9)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, color=INK, fontsize=10, loc="left", pad=8)
    if xlabel:
        ax.set_xlabel(xlabel, color=INK_2, fontsize=8)
    if ylabel:
        ax.set_ylabel(ylabel, color=INK_2, fontsize=8)
    return ax


def fig_yaw_sweep(candidates, chosen_yaw, path=None):
    """Overlap and height-correlation vs yaw.

    Both series are dimensionless and share the -1..1 range, so they go on ONE
    axis -- the point of the figure is precisely that overlap alone is flat
    across the 180 deg pair while the height channel is not.
    """
    import matplotlib.pyplot as plt

    yaw = np.array([c["yaw"] for c in candidates])
    ov = np.array([c["overlap"] for c in candidates])
    hc = np.array([c["hcorr"] for c in candidates])
    o = np.argsort(yaw)
    yaw, ov, hc = yaw[o], ov[o], hc[o]

    fig, ax = plt.subplots(figsize=(9, 3.6), facecolor=SURFACE)
    _style(ax, "Yaw search: footprint overlap cannot separate the 180° pair; height correlation can",
           "yaw about vertical (degrees, grid north)", "score (dimensionless, −1…1)")
    ax.axhline(0, color=MUTED, linewidth=0.8)
    ax.plot(yaw, ov, color=C1, linewidth=2, label="footprint overlap")
    ax.plot(yaw, hc, color=C2, linewidth=2, label="height correlation")
    opp = (chosen_yaw + 180) % 360
    for x, lab, col in ((chosen_yaw, f"chosen {chosen_yaw:.0f}°", C1),
                        (opp, f"rejected {opp:.0f}°", C8)):
        ax.axvline(x, color=col, linewidth=1.2, linestyle="--", alpha=0.9)
        # inside the axes, not above them -- at y=1.02 this collided with the title
        ax.annotate(lab, (x, 0.94), xycoords=("data", "axes fraction"),
                    color=col, fontsize=8, ha="center",
                    bbox=dict(facecolor=SURFACE, edgecolor="none", pad=1.5))
    ax.set_xlim(0, 360)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticks(range(0, 361, 45))
    leg = ax.legend(frameon=False, fontsize=8, loc="lower right", ncol=2)
    for t in leg.get_texts():
        t.set_color(INK_2)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, facecolor=SURFACE)
    return fig


def fig_alignment(dsm, target, mesh_xy_before, mesh_xy_after, path=None):
    """Mesh footprint against the DSM target, before and after solving."""
    import matplotlib.pyplot as plt

    building = target["building"]
    ys, xs = np.nonzero(building)
    r0, r1 = ys.min() - 25, ys.max() + 26
    c0, c1 = xs.min() - 25, xs.max() + 26
    sub = dsm.data.astype(float)[r0:r1, c0:c1]
    e0, n1 = dsm.xy(np.array([r0]), np.array([c0]))
    e1, n0 = dsm.xy(np.array([r1]), np.array([c1]))
    extent = [float(e0[0]), float(e1[0]), float(n0[0]), float(n1[0])]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), facecolor=SURFACE)
    for ax, xy, ttl in ((axes[0], mesh_xy_before, "Before: ARKit frame (arbitrary yaw & origin)"),
                        (axes[1], mesh_xy_after, "After: solved 4-DOF transform")):
        _style(ax, ttl, "easting (m, EPSG:32643)", "northing (m)")
        ax.imshow(sub, extent=extent, origin="upper", cmap="Greys_r",
                  alpha=0.85, aspect="equal")
        cs = ax.contour(np.linspace(extent[0], extent[1], sub.shape[1]),
                        np.linspace(extent[3], extent[2], sub.shape[0]),
                        building[r0:r1, c0:c1].astype(float), levels=[0.5],
                        colors=[C3], linewidths=2)
        ax.scatter(xy[:, 0], xy[:, 1], s=0.4, c=C2, alpha=0.35, linewidths=0,
                   label="mesh footprint")
        ax.plot([], [], color=C3, linewidth=2, label="DSM building target")
        # These axes carry a dark raster underneath, so an unframed legend is
        # unreadable; give it the chart surface as a backing plate.
        leg = ax.legend(fontsize=8, loc="upper right", framealpha=0.92,
                        facecolor=SURFACE, edgecolor=GRID)
        for t in leg.get_texts():
            t.set_color(INK)
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, facecolor=SURFACE)
    return fig


def fig_segmentation(centroids, planes, geoms, deck_z, path=None,
                     min_terrace_area=3.0):
    """Plan view coloured by surface class, plus the terrace elevation ladder."""
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.4), facecolor=SURFACE,
                             gridspec_kw={"width_ratios": [1.35, 1]})

    ax = _style(axes[0], "Segmented surfaces (plan view)",
                "easting (m)", "northing (m)")
    seen = set()
    for p, g in zip(planes, geoms):
        C = centroids[p["faces"]]
        cls = g["orientation"]
        ax.scatter(C[:, 0], C[:, 1], s=1.1, c=CLASS_COLORS[cls], linewidths=0,
                   alpha=0.75, label=cls if cls not in seen else None)
        seen.add(cls)
    ax.set_aspect("equal")
    leg = ax.legend(frameon=False, fontsize=8, loc="upper left",
                    markerscale=8, title="surface class")
    leg.get_title().set_color(INK_2)
    leg.get_title().set_fontsize(8)
    for t in leg.get_texts():
        t.set_color(INK_2)

    ax = _style(axes[1], f"Horizontal terraces by elevation (≥ {min_terrace_area:g} m²)",
                "plane area (m²)", "height above deck (m)")
    # Threshold matches the one the object hierarchy uses. Plotting every
    # sliver instead produces a pile of colliding labels around +2.5 m.
    terr = [(p, g) for p, g in zip(planes, geoms)
            if g["orientation"] == "horizontal" and p["area_m2"] >= min_terrace_area]
    terr.sort(key=lambda pg: pg[0]["centroid"][2])
    ys = [p["centroid"][2] - deck_z for p, _ in terr]
    ws = [p["area_m2"] for p, _ in terr]
    ax.barh(ys, ws, height=0.13, color=C1, edgecolor=SURFACE, linewidth=1.5)
    for y, w in zip(ys, ws):
        ax.annotate(f"{w:.1f} m²  ·  +{y:.2f} m", (w, y), xytext=(6, 0),
                    textcoords="offset points", va="center", fontsize=8,
                    color=INK_2)
    ax.set_xlim(0, max(ws) * 1.5)
    ax.set_ylim(-0.4, max(ys) + 0.5)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, facecolor=SURFACE)
    return fig


def fig_solar(horizon, usable, path=None):
    """Horizon obstruction (polar) and the usable-area map."""
    import matplotlib.pyplot as plt

    fig = plt.figure(figsize=(11.5, 5.2), facecolor=SURFACE)
    ax = fig.add_subplot(1, 2, 1, projection="polar")
    ax.set_facecolor(SURFACE)
    az = np.radians([h["true_azimuth_deg"] for h in horizon] +
                    [horizon[0]["true_azimuth_deg"]])
    el = np.array([h["horizon_elev_deg"] for h in horizon] +
                  [horizon[0]["horizon_elev_deg"]])
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.plot(az, el, color=C2, linewidth=2)
    ax.fill(az, el, color=C2, alpha=0.18)
    ax.set_title("Horizon obstruction from deck centre\n(true-north azimuth, degrees elevation)",
                 color=INK, fontsize=10, pad=14)
    ax.tick_params(colors=INK_2, labelsize=8)
    ax.grid(color=GRID, linewidth=0.6)
    ax.set_rlabel_position(135)
    ax.spines["polar"].set_color(GRID)

    ax2 = _style(fig.add_subplot(1, 2, 2), None, "easting (m)", "northing (m)")
    m = usable["_mask"]
    e0, n1 = usable["_origin"]
    c = usable["_cell"]
    extent = [e0, e0 + m.shape[1] * c, n1 - m.shape[0] * c, n1]
    ax2.imshow(np.where(m, 1.0, np.nan), extent=extent, origin="upper",
               cmap=_mono_cmap(C1), vmin=0, vmax=1, aspect="equal")
    ax2.set_title(
        f"Deck area available for panels — {usable['usable_area_m2']:.1f} m² "
        f"of {usable['gross_area_m2']:.1f} m² gross",
        color=INK, fontsize=10, loc="left", pad=8)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150, facecolor=SURFACE)
    return fig


def _mono_cmap(hex_color):
    from matplotlib.colors import LinearSegmentedColormap
    return LinearSegmentedColormap.from_list("m", [hex_color, hex_color])

In [ ]:
%%writefile rooflib/export_schema.py
"""Emit the pipeline result in the target site-analysis JSON schema.

Schema notes that drove decisions here:

* Polygons are lat/lng closed rings, so plane footprints are traced in UTM,
  simplified, then reprojected -- simplifying after reprojection would apply a
  metre-based tolerance to degrees.
* `azimuth_deg` is compass (0 = N, 90 = E, 180 = S). Our planes carry TRUE-north
  azimuths already, which is what compass means; grid azimuths would be wrong
  by the grid convergence.
* `base_height_m` is above ground level, not above the deck and not an absolute
  elevation, so it is measured from a ground datum derived from the DSM.
* Only non-vertical planes are roof faces. Walls are structure, not roof.
"""
from __future__ import annotations

import numpy as np

# The schema's obstruction vocabulary is vent | ac_unit | chimney | skylight |
# pipe | unknown. Our classifier speaks a different language (it was built for
# Indian rooftops: water tanks, stair headrooms, terrace blocks). Only map where
# the correspondence is defensible; everything else is honestly "unknown"
# rather than forced into a bucket a downstream consumer would misread.
TYPE_MAP = {
    "water_tank": "unknown",
    "water_tank_elevated": "unknown",
    "stair_headroom": "unknown",
    "superstructure_block": "unknown",
    "raised_platform": "unknown",
    "small_object": "unknown",
    "unclassified_object": "unknown",
}


def _rings_from_mask(mask, origin, cell):
    """Trace closed boundary rings of a binary mask, in UTM metres."""
    import matplotlib
    matplotlib.use("Agg", force=False)
    import matplotlib.pyplot as plt

    e0, n1 = origin
    H, W = mask.shape
    xs = e0 + (np.arange(W) + 0.5) * cell
    ys = n1 - (np.arange(H) + 0.5) * cell
    fig = plt.figure()
    try:
        cs = plt.contour(xs, ys, mask.astype(float), levels=[0.5])
        rings = []
        for path in cs.get_paths():
            v = path.vertices
            if len(v) >= 4:
                rings.append(np.asarray(v, float))
    finally:
        plt.close(fig)
    rings.sort(key=lambda r: -_ring_area(r))
    return rings


def _ring_area(r):
    x, y = r[:, 0], r[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def _douglas_peucker(pts, eps):
    """Simplify a polyline; keeps ring vertex counts sane for JSON."""
    if len(pts) < 3:
        return pts
    start, end = pts[0], pts[-1]
    d = end - start
    n = np.linalg.norm(d)
    if n < 1e-12:
        dist = np.linalg.norm(pts - start, axis=1)
    else:
        # 2-D cross product written out: numpy 2.0 removed the 2-vector form
        v = pts - start
        dist = np.abs(d[0] * v[:, 1] - d[1] * v[:, 0]) / n
    i = int(np.argmax(dist))
    if dist[i] > eps:
        left = _douglas_peucker(pts[: i + 1], eps)
        right = _douglas_peucker(pts[i:], eps)
        return np.vstack([left[:-1], right])
    return np.vstack([start, end])


def plane_fit_stats(plane, verts_enu, faces, inlier_tol=0.05):
    """RMSE / inlier ratio / sample count of the plane fit, from real vertices."""
    vid = np.unique(faces[plane["faces"]].ravel())
    P = verts_enu[vid]
    n = np.asarray(plane["normal"], float)
    d = (P - np.asarray(plane["centroid"], float)) @ n
    rmse = float(np.sqrt(np.mean(d ** 2)))
    inlier = float(np.mean(np.abs(d) <= inlier_tol))
    return {"rmse_m": round(rmse, 4),
            "inlier_ratio": round(inlier, 4),
            "sample_count": int(len(P))}


def _face_confidence(fit, area_m2):
    """Blend fit quality with how much evidence the plane has.

    A 0.9 m^2 sliver fitted perfectly is not as trustworthy as a 40 m^2 plane
    fitted well, so area gates the score rather than only the residual.
    """
    q = fit["inlier_ratio"] * float(np.exp(-fit["rmse_m"] / 0.05))
    size = min(1.0, area_m2 / 10.0)
    return round(float(np.clip(0.35 + 0.65 * q * size, 0.0, 0.99)), 3)


def build_site_json(*, address_id, planes, geoms, plane_records, terraces,
                    object_records, verts_enu, faces, centroids, dsm, mask,
                    crs_wkt, to_lonlat, ground_elev_m, deck_z,
                    footprint_mask_fn, imagery_date=None,
                    dsm_resolution_m=0.1, provider="iphone-lidar+dsm",
                    simplify_m=0.12, min_face_area_m2=1.0,
                    max_fit_rmse_m=0.20, min_inlier_ratio=0.45,
                    extra_warnings=()):
    """Assemble the schema document."""
    # ---- roof faces: everything that is not a wall -------------------------
    face_ids = {}
    faces_out = []
    face_anchor = []          # (face_id, e, n, z) for attributing obstructions
    rejected = []
    idx = 0
    for p, g, rec in zip(planes, geoms, plane_records):
        if g["orientation"] == "vertical":
            continue
        if p["area_m2"] < min_face_area_m2:
            continue
        # Quality gate. Residual-absorption can leave a "plane" whose members
        # span metres; emitting those as roof faces would hand a consumer a
        # confident-looking surface that does not exist.
        fit_pre = plane_fit_stats(p, verts_enu, faces)
        if (fit_pre["rmse_m"] > max_fit_rmse_m
                or fit_pre["inlier_ratio"] < min_inlier_ratio):
            rejected.append((p["plane_id"], p["area_m2"], fit_pre))
            continue
        idx += 1
        fid = f"face-{idx}"
        face_ids[p["plane_id"]] = fid

        m_, orig_, cell_ = footprint_mask_fn(centroids[p["faces"]][:, :2], cell=0.15)
        rings = _rings_from_mask(m_, orig_, cell_)
        if not rings:
            polygon = []
        else:
            r = _douglas_peucker(rings[0], simplify_m)
            if not np.allclose(r[0], r[-1]):
                r = np.vstack([r, r[0]])
            lon, lat = to_lonlat(r[:, 0], r[:, 1], crs_wkt)
            lon = np.atleast_1d(lon); lat = np.atleast_1d(lat)
            polygon = [[round(float(a), 7), round(float(o), 7)]
                       for a, o in zip(lat, lon)]

        face_anchor.append((fid, float(p["centroid"][0]), float(p["centroid"][1]),
                            float(p["centroid"][2])))
        fit = fit_pre
        tilt = float(g["tilt_deg"])
        az = g["true_azimuth_deg"]
        faces_out.append({
            "id": fid,
            "polygon": polygon,
            "tilt_deg": round(tilt, 2),
            "azimuth_deg": None if not np.isfinite(az) else round(float(az), 2),
            "area_m2": round(float(p["area_m2"]), 2),
            "base_height_m": round(float(p["z_min"] - ground_elev_m), 2),
            "plane_fit": fit,
            "confidence": _face_confidence(fit, p["area_m2"]),
        })

    # ---- obstructions ------------------------------------------------------
    obs_out = []
    for i, o in enumerate(object_records, start=1):
        lon, lat = to_lonlat(np.array([o["centroid_e"]]),
                             np.array([o["centroid_n"]]), crs_wkt)
        on_face = face_ids.get(o.get("on_terrace_plane_id"), None)
        if on_face is None and face_anchor:
            # The terrace this object sits on may have been rejected by the fit
            # gate. Fall back to the nearest surviving face that is actually
            # below the object, rather than dropping the association.
            base = float(o["base_elev_m"])
            cands = [a for a in face_anchor if a[3] <= base + 0.5] or face_anchor
            on_face = min(cands, key=lambda a: np.hypot(a[1] - o["centroid_e"],
                                                        a[2] - o["centroid_n"]))[0]
            o = {**o, "_attributed_by": "nearest_face_fallback"}
        obs_out.append({
            "id": f"obs-{i}",
            "type": TYPE_MAP.get(o["label"], "unknown"),
            "source_label": o["label"],          # our richer label, kept alongside
            "centroid": {"lat": round(float(np.atleast_1d(lat)[0]), 7),
                         "lng": round(float(np.atleast_1d(lon)[0]), 7)},
            "footprint_m": {"width": round(float(o["footprint_bbox_m"][0]), 2),
                            "length": round(float(o["footprint_bbox_m"][1]), 2)},
            "height_m": round(float(o["top_elev_m"] - o["base_elev_m"]), 2),
            "on_face": on_face,
            "confidence": round(float(o["label_confidence"]), 2),
            **({"attributed_by": "nearest_face_fallback"}
               if o.get("_attributed_by") else {}),
        })

    # ---- warnings: real ones, derived from the data ------------------------
    warnings = list(extra_warnings)
    if rejected:
        area = sum(r[1] for r in rejected)
        warnings.append(
            f"planes_rejected_on_fit_quality: {len(rejected)} candidate faces "
            f"({area:.1f} m2) failed rmse<={max_fit_rmse_m} m / "
            f"inlier>={min_inlier_ratio}; these are fusion noise, not surfaces")
    unplaced = [o["id"] for o in obs_out if o["on_face"] is None]
    if unplaced:
        warnings.append(f"obstructions_not_attributed_to_a_face: {','.join(unplaced)}")
    if any(f["polygon"] == [] for f in faces_out):
        warnings.append("polygon_trace_failed_on_one_or_more_faces")
    if all(o["type"] == "unknown" for o in obs_out) and obs_out:
        warnings.append(
            "obstruction_types_unmapped: classifier vocabulary (water_tank, "
            "stair_headroom, superstructure_block) does not correspond to the "
            "schema's vent/ac_unit/chimney/skylight/pipe set; see source_label")

    lon_c, lat_c = to_lonlat(np.array([float(np.mean(verts_enu[:, 0]))]),
                             np.array([float(np.mean(verts_enu[:, 1]))]), crs_wkt)

    return {
        "address_id": address_id,
        "location": {"lat": round(float(np.atleast_1d(lat_c)[0]), 7),
                     "lng": round(float(np.atleast_1d(lon_c)[0]), 7)},
        "source": {
            "imagery_date": imagery_date,
            "dsm_resolution_m": dsm_resolution_m,
            "provider": provider,
        },
        "roof_faces": faces_out,
        "obstructions": obs_out,
        "trees": [],
        "warnings": warnings,
    }

In [ ]:
import importlib, rooflib
for m in ["io_utils", "georef", "segment", "solar", "exporters", "viz",
          "export_schema"]:
    importlib.import_module(f"rooflib.{m}")
    importlib.reload(sys.modules[f"rooflib.{m}"])

from rooflib.io_utils import read_raster, load_obj_from_zip, Raster
from rooflib import georef as G, segment as S, solar as SO, exporters as X, viz as V
from rooflib import export_schema as ES
print("rooflib ready")

## 3 · Load and characterise the inputs

In [ ]:
dsm  = read_raster(INPUT_DIR / "dsm.tif")
rgb  = read_raster(INPUT_DIR / "rgb.tif")
mask = read_raster(INPUT_DIR / "building_mask.tif")
mesh, mesh_meta = load_obj_from_zip(INPUT_DIR / "lidar_object.zip")

assert dsm.transform == mask.transform == rgb.transform, \
    "rasters are not on a common grid - the pipeline assumes they are co-registered"

res = dsm.res[0]
print(f"rasters   : {dsm.shape[1]}x{dsm.shape[0]} px @ {res} m  (co-registered: identical geotransform)")
print(f"            extent {dsm.shape[1]*res:.1f} x {dsm.shape[0]*res:.1f} m")
print(f"DSM       : {np.nanmin(dsm.data):.2f} .. {np.nanmax(dsm.data):.2f} m   nodata={dsm.nodata}")
print(f"mask      : {(mask.data==1).sum()} px = {(mask.data==1).sum()*res*res:.1f} m^2")
print(f"mesh      : {mesh_meta['n_vertices']} verts, {mesh_meta['n_faces']} faces, "
      f"{mesh.face_areas().sum():.1f} m^2 surface")
print(f"            bbox {np.round(mesh.V.max(0)-mesh.V.min(0), 2)} m (ARKit frame)")
print(f"            {len(mesh.materials)} texture atlases, {len(np.unique(mesh.face_group))} 'o' groups")

In [ ]:
# --- structural facts that determine how segmentation has to work -----------
shells = mesh.connected_shells()
n_shells = len(np.unique(shells))

grp_extent = {}
for g in np.unique(mesh.face_group):
    C = mesh.face_centroids()[mesh.face_group == g]
    grp_extent[g] = (np.ptp(C[:, 0]), np.ptp(C[:, 2]))

print(f"connected shells: {n_shells}")
print("  -> the mesh is NOT one surface; topological connectivity carries no semantics.")
print(f"\n'o' groups and their horizontal extents (ARKit x,z):")
for g, e in grp_extent.items():
    print(f"  {g}: {e[0]:5.1f} x {e[1]:5.1f} m")
print("  -> every group spans the whole roof: these are texture atlases, not parts.")

N, A = mesh.face_normals(), mesh.face_areas()
tilt = np.degrees(np.arccos(np.clip(np.abs(N[:, 1]), 0, 1)))
noise = A[(tilt > 5) & (tilt < 85)].sum()
print(f"\nfaces between 5 and 85 deg off-axis: {noise:.1f} m^2 ({100*noise/A.sum():.0f}% of surface)")
print("  -> transition/noise triangles from mesh fusion; smoothed before plane fitting.")

In [ ]:
# --- the mask overshoots the roof: quantify before relying on it ------------
target = G.build_target(dsm, mask)
print(f"building_mask polygon      : {target['mask_area_m2']:8.1f} m^2")
print(f"  of which real roof surface: {target['building_area_m2']:8.1f} m^2")
print(f"  discarded (>2 m below roof): {target['discarded_area_m2']:8.1f} m^2"
      f"  ({100*target['discarded_area_m2']/target['mask_area_m2']:.0f}% of the mask)")
print(f"roof median elevation      : {target['roof_median']:8.2f} m")
print("\n-> the mask is a coarse rotated rectangle, usable as a region of interest")
print("   but NOT as the roof footprint. The alignment target is mask AND DSM.")

## 4 · Stage 2 — georeferencing

Four unknowns, not seven: ARKit's `+y` is gravity-up, and ARKit is metric, so
roll and pitch are already solved and only **yaw, E, N, Z** (plus an optional
scale check) remain.

Translation is not searched — at each trial yaw it is solved in closed form by
FFT cross-correlation, which is why a full 360° sweep is cheap.

In [ ]:
deck_local, deck_info = G.estimate_deck_level(mesh)
deck_n, deck_tilt, deck_area = G.deck_normal(mesh, deck_local)

print(f"deck (walking surface) at ARKit y = {deck_local:.3f}, {deck_info['deck_area_m2']:.1f} m^2")
print(f"deck normal {np.round(deck_n, 5)}  ->  {deck_tilt:.3f} deg from vertical")
print("\nGRAVITY CHECK: ARKit's +y is within a degree of true vertical, so roll/pitch")
print("are already solved and this is a 4-DOF problem.")
print("\nhorizontal surface bands in the ARKit frame (area-weighted):")
for b in deck_info["bands"]:
    print(f"  y = {b['y']:6.2f}   {b['area_m2']:6.1f} m^2")

In [ ]:
pts, pt_normals, pt_w, pt_face = mesh.sample_surface_with_normals(n_per_face=10)
print(f"sampled {len(pts):,} surface points (barycentric, so top-down rasters are solid)")

hand = G.handedness_check(pts, target, dsm, deck_local, yaw_step=2.0)
rh, mi = hand["right_handed"], hand["mirrored"]
print("\n--- CHIRALITY, decided by data rather than convention ---")
print(f"  east=+x, north=-z  : yaw {rh['yaw']:5.1f}  overlap {rh['overlap']:.3f}  "
      f"height-corr {rh['hcorr']:+.3f}  joint {rh['joint']:.3f}")
print(f"  mirrored (north=+z): yaw {mi['yaw']:5.1f}  overlap {mi['overlap']:.3f}  "
      f"height-corr {mi['hcorr']:+.3f}  joint {mi['joint']:.3f}")
print(f"  verdict: {hand['verdict']}  (margin {hand['margin']:.3f})")
assert hand["verdict"] == "right_handed", "chirality check failed - inspect before continuing"

In [ ]:
candidates, _ = G.solve_yaw(pts, target, dsm, deck_local, yaw_step=1.0)
winner, ambiguity = G.disambiguate_180(candidates)

print("--- top 5 by footprint overlap ---")
for c in candidates[:5]:
    print(f"  yaw {c['yaw']:5.1f}   overlap {c['overlap']:.4f}   height-corr {c['hcorr']:+.4f}")

print("\n--- THE 180 DEGREE DECISION ---")
p, o = ambiguity["primary"], ambiguity["opposite"]
print(f"  yaw {p['yaw']:5.1f} : overlap {p['overlap']:.4f}   height-corr {p['hcorr']:+.4f}")
print(f"  yaw {o['yaw']:5.1f} : overlap {o['overlap']:.4f}   height-corr {o['hcorr']:+.4f}")
print(f"\n  overlap separates them by only {ambiguity['overlap_margin']:.4f}  <- not decisive")
print(f"  height  separates them by       {ambiguity['hcorr_margin']:.4f}  <- decisive")
print(f"  chosen yaw = {ambiguity['chosen_yaw']:.1f} deg (grid)")

In [ ]:
V.fig_yaw_sweep(candidates, ambiguity["chosen_yaw"],
                path=OUTPUT_DIR / "qc_yaw_sweep.png")
import matplotlib.pyplot as plt; plt.show()

In [ ]:
# Scale, measured with a SYMMETRIC metric. An overlap ratio normalised by
# min(source, target) is biased toward shrinking the source and will happily
# report a bogus sub-1.0 scale; IoU has no such bias.
scale_rows = G.scale_iou(pts, target, dsm, deck_local, ambiguity["chosen_yaw"],
                         [0.94, 0.96, 0.98, 1.00, 1.02, 1.04, 1.06])
print("scale   IoU")
for r in scale_rows:
    bar = "#" * int((r["iou"] - 0.80) * 200)
    print(f" {r['scale']:.2f}   {r['iou']:.4f}  {bar}")
best_scale = max(scale_rows, key=lambda r: r["iou"])["scale"]
print(f"\nIoU peak near scale {best_scale:.2f}; the refinement below fits it properly.")

In [ ]:
solution = G.refine(pts, pt_normals, target, dsm, deck_local,
                    ambiguity["chosen_yaw"], winner["tE"], winner["tN"],
                    fit_scale=True)
M = G.matrix_from_solution(solution)

print(f"yaw   {solution['yaw_deg']:10.3f} deg (grid)")
print(f"scale {solution['scale']:10.4f}")
print(f"tE    {solution['tE']:10.3f} m")
print(f"tN    {solution['tN']:10.3f} m")
print(f"tZ    {solution['tZ']:10.3f} m")
print(f"cost  {solution['cost_start']:.4f} -> {solution['cost']:.4f} in {solution['n_iter']} iterations")
print("\n4x4 ARKit -> UTM 43N:")
print(np.array2string(M, precision=5, suppress_small=True))

In [ ]:
check = G.verify_alignment(mesh, M, target, dsm)
print("--- independent verification ---")
for k, v in check.items():
    print(f"  {k:26s} {v:.4f}" if isinstance(v, float) else f"  {k:26s} {v}")

deck_z = check["deck_mean_elev_m"]
print(f"\n  mesh deck lands at {deck_z:.2f} m; DSM roof median is {target['roof_median']:.2f} m"
      f"  (difference {abs(deck_z-target['roof_median']):.2f} m)")

# the deck normal must still be vertical after transforming - a transposed
# normal transform is silent otherwise
Nl, Cl, Al = mesh.face_normals(), mesh.face_centroids(), mesh.face_areas()
sel = (Nl[:, 1] > 0.98) & (np.abs(Cl[:, 1] - deck_local) < 0.15)
nw = G.apply_normals(M, Nl[sel]); nw = (Al[sel, None] * nw).sum(0); nw /= np.linalg.norm(nw)
print(f"  deck normal in ENU {np.round(nw, 5)} -> {np.degrees(np.arccos(np.clip(nw[2],-1,1))):.3f} deg from vertical")

In [ ]:
before = (mesh.V @ G.AXIS_MAP.T)[:, :2] + np.array([solution["tE"], solution["tN"]])
after  = G.apply(M, mesh.V)[:, :2]
V.fig_alignment(dsm, target, before, after, path=OUTPUT_DIR / "qc_alignment.png")
plt.show()

In [ ]:
lon, lat = G.to_lonlat(np.array([solution["tE"]]), np.array([solution["tN"]]), dsm.crs_wkt)
lon, lat = float(np.atleast_1d(lon)[0]), float(np.atleast_1d(lat)[0])
convergence = G.grid_convergence(lon, lat, G.utm_zone_center(dsm.crs_wkt))

print(f"building centre : {lat:.6f} N, {lon:.6f} E")
print(f"grid convergence: {convergence:+.4f} deg")
print(f"  true_azimuth = grid_azimuth {convergence:+.4f}")
print("  every azimuth exported below is TRUE north referenced.")

tiltinfo = G.residual_tilt(G.apply(M, pts), G.apply_normals(M, pt_normals), dsm, target)
print("\nroof plane, mesh vs DSM:")
for k, v in tiltinfo.items():
    print(f"  {k:32s} {v:8.3f}")
print("\n  both are essentially flat; the disagreement is within DSM noise")
print("  (the DSM's own roof-plane residual std is ~0.39 m over a flat slab),")
print("  so no tilt correction is applied and the mesh keeps ARKit's gravity.")

## 5 · Stage 3 — segmentation

Normals are smoothed **bilaterally** first. Plain isotropic smoothing is actively
harmful here: at every deck/parapet corner it averages a horizontal and a vertical
normal into a spurious 45° one and measurably *grows* the noise band.

Planes are then found by normal-direction clustering followed by an offset
histogram, rather than brute-force RANSAC — far cheaper on ~90k faces, and
buildings are made of a handful of normal directions.

In [ ]:
Cw = G.apply(M, mesh.face_centroids())
Nw = G.apply_normals(M, mesh.face_normals())
Aw = mesh.face_areas() * solution["scale"] ** 2

t_raw = np.degrees(np.arccos(np.clip(np.abs(Nw[:, 2]), 0, 1)))
Ns = S.smooth_normals(Cw, Nw, Aw, normal_sigma_deg=15, iterations=3, radius=0.30)
t_sm = np.degrees(np.arccos(np.clip(np.abs(Ns[:, 2]), 0, 1)))

for tag, t in (("raw", t_raw), ("bilateral", t_sm)):
    nb = Aw[(t > 5) & (t < 70)].sum()
    print(f"  {tag:10s} noise band (5-70 deg): {nb:6.1f} m^2  ({100*nb/Aw.sum():.0f}%)")

In [ ]:
planes = S.extract_planes(Cw, Ns, Aw, angle_tol_deg=12, offset_bin=0.12, min_area=0.8)
assigned = np.zeros(len(Aw), bool)
for p in planes:
    assigned[p["faces"]] = True
print(f"direction+offset clustering : {len(planes)} planes, {100*Aw[assigned].sum()/Aw.sum():.0f}% of area")

planes, assigned = S.assign_residual_faces(planes, Cw, Ns, Aw, assigned)
planes.sort(key=lambda p: -p["area_m2"])
for i, p in enumerate(planes):
    p["plane_id"] = i
    z = Cw[p["faces"]][:, 2]
    p["z_min"], p["z_max"] = float(z.min()), float(z.max())
geoms = [S.plane_geometry(p, convergence) for p in planes]
print(f"after residual assignment   : {len(planes)} planes, {100*Aw[assigned].sum()/Aw.sum():.0f}% of area")

deck_plane = planes[0]
deck_z = float(deck_plane["centroid"][2])
print(f"\ndeck plane: #{deck_plane['plane_id']}, {deck_plane['area_m2']:.1f} m^2 at z = {deck_z:.2f} m")

In [ ]:
print(f"{'id':>3} {'area m2':>8} {'class':<11} {'tilt':>6} {'trueAz':>8} {'aspect':>7} {'z':>7} {'+deck':>7}")
for p, g in list(zip(planes, geoms))[:18]:
    az = "  flat" if not np.isfinite(g["grid_azimuth_deg"]) else f"{g['true_azimuth_deg']:8.1f}"
    asp = SO._compass(g["true_azimuth_deg"]) if np.isfinite(g["grid_azimuth_deg"]) else "flat"
    print(f"{p['plane_id']:3d} {p['area_m2']:8.2f} {g['orientation']:<11} {g['tilt_deg']:6.1f} "
          f"{az} {asp:>7} {p['centroid'][2]:7.2f} {p['centroid'][2]-deck_z:+7.2f}")

In [ ]:
# Wall azimuths are an independent check on yaw: a rectilinear building must
# produce four families 90 deg apart, and they must be consistent.
wall_az = sorted(g["grid_azimuth_deg"] for p, g in zip(planes, geoms)
                 if g["orientation"] == "vertical" and p["area_m2"] > 2)
families = []
for a in wall_az:
    if families and a - families[-1][-1] < 20:
        families[-1].append(a)
    else:
        families.append([a])
main = [f for f in families if len(f) >= 2]
singles = [f for f in families if len(f) == 1]
means = [float(np.mean(f)) for f in main]

print("wall azimuth families (grid):")
for f, m in zip(main, means):
    print(f"  {m:6.1f} deg   n={len(f):2d}   spread {max(f)-min(f):.1f} deg")
for f in singles:
    print(f"  {f[0]:6.1f} deg   n= 1   (isolated: small plane or fusion noise)")

spac = [means[i+1] - means[i] for i in range(len(means) - 1)]
print("\nspacings between families:", [f"{s:.1f}" for s in spac])
if len(means) == 4 and all(abs(s - 90) < 3 for s in spac):
    print("\n-> four families within 3 deg of orthogonal. The building is rectilinear,")
    print("   and this INDEPENDENTLY confirms the solved yaw: a wrong yaw would still")
    print("   produce orthogonal families, but a wrong CHIRALITY or a mis-fitted scale")
    print("   would smear them - so this is a genuine consistency check on stage 2.")
else:
    print("\n-> expected four families ~90 deg apart on a rectilinear building; inspect.")

In [ ]:
terraces = S.extract_objects_hierarchical(planes, geoms, Cw, Aw, Ns)

print("Objects are peeled off one terrace at a time, highest first.")
print("A single 'everything above the deck' clustering fails twice over here:")
print("the parapet ring bridges separate objects, and the roof is layered")
print("(deck -> a ~3 m block -> tanks standing on that block).\n")

object_records = []
for t in terraces:
    rel = t["terrace_z"] - deck_z
    print(f"TERRACE plane#{t['terrace_plane_id']}  z={t['terrace_z']:.2f} ({rel:+.2f} above deck)"
          f"  {t['terrace_area_m2']:.1f} m^2  ->  {len(t['objects'])} object(s)")
    for sel in t["objects"]:
        info = S.classify_object(None, sel, Cw, Ns, Aw, t["terrace_z"])
        info["on_terrace_plane_id"] = t["terrace_plane_id"]
        info["on_terrace_z"] = t["terrace_z"]
        object_records.append(info)
        print(f"    {info['label']:<26s} {info['surface_area_m2']:6.1f} m^2  "
              f"{info['footprint_bbox_m'][0]:4.1f}x{info['footprint_bbox_m'][1]:4.1f} m  "
              f"top z {info['top_elev_m']:6.2f}  conf {info['label_confidence']:.2f}")
print("\nLabels are heuristic: on a noisy phone scan a stair headroom and a walled")
print("tank plinth have overlapping geometric signatures, so ambiguous clusters")
print("stay generic rather than asserting a type downstream code would trust.")

In [ ]:
V.fig_segmentation(Cw, planes, geoms, deck_z, path=OUTPUT_DIR / "qc_segmentation.png")
plt.show()

## 6 · Stage 4 — solar attributes

In [ ]:
plane_records = [SO.plane_solar_record(p, g, Cw, Aw, deck_z, convergence)
                 for p, g in zip(planes, geoms)]

deck_objects = [sel for t in terraces if abs(t["terrace_z"] - deck_z) < 0.05
                for sel in t["objects"]]
usable = SO.usable_area(deck_plane, Cw, deck_objects,
                        edge_setback_m=0.6, obstruction_clearance_m=0.5)

print("Deck area budget")
print(f"  gross plane footprint      {usable['gross_area_m2']:8.2f} m^2")
print(f"  after {usable['edge_setback_m']} m edge setback     {usable['after_setback_m2']:8.2f} m^2")
print(f"  blocked by roof objects    {usable['blocked_by_objects_m2']:8.2f} m^2")
print(f"  USABLE for panels          {usable['usable_area_m2']:8.2f} m^2")
print(f"\n  -> {100*usable['usable_area_m2']/usable['gross_area_m2']:.0f}% of the gross deck is actually mountable.")

In [ ]:
sun = SO.solar_position_summary(lat, lon)
print(f"latitude {sun['latitude_deg']:.4f} ({sun['hemisphere']} hemisphere)")
print(f"panels should face TRUE azimuth {sun['optimal_azimuth_true_deg']:.0f} deg")
print(f"rule-of-thumb fixed tilt        {sun['optimal_fixed_tilt_deg_rule_of_thumb']:.0f} deg")
for k in ("summer_solstice", "equinox", "winter_solstice"):
    s = sun[k]
    print(f"  {k:18s} noon elevation {s['solar_noon_elevation_deg']:5.1f} deg   "
          f"shadow {s['shadow_length_per_m']:.2f} m per m of height")

In [ ]:
horizon = SO.horizon_profile(dsm, deck_plane["centroid"][0], deck_plane["centroid"][1],
                             deck_z, n_sectors=36, convergence_deg=convergence)
sectors = SO.sector_summary(dsm, mask, target, deck_z, convergence_deg=convergence)

worst = max(horizon, key=lambda h: h["horizon_elev_deg"])
print(f"worst horizon obstruction: {worst['horizon_elev_deg']:.1f} deg at true azimuth "
      f"{worst['true_azimuth_deg']:.0f}, {worst['distance_m']} m away, "
      f"{worst['obstruction_height_m']:+.2f} m above the deck")

far = max(sectors, key=lambda s: s["elev_angle_deg"])
print(f"worst NEIGHBOUR (beyond 8 m): {far['elev_angle_deg']:.1f} deg at "
      f"{far['distance_at_max_m']:.0f} m, {far['above_deck_m']:+.2f} m above deck")
print("\n-> on this roof the dominant shading is the building's OWN superstructure,")
print("   not the neighbours. Panel layout must work around it.")
print(f"\ninter-row spacing for a 3 m obstruction at 25 deg sun: "
      f"{SO.interrow_spacing(3.0, 25.0):.2f} m")

In [ ]:
V.fig_solar(horizon, usable, path=OUTPUT_DIR / "qc_solar.png")
plt.show()

## 7 · Stage 5 — export

Geometry is written twice:

* **local ENU** — origin at the building centroid. Most 3D viewers store vertex
  positions as float32; at UTM easting 250811 / northing 2546703 the float32 step
  is already ~0.25 m, so an absolute-coordinate mesh visibly jitters and z-fights.
* **absolute UTM 43N** — for GIS, which stores doubles and does not have that problem.

`transform.json` carries the matrix and the origin, so either can be recovered
from the other.

In [ ]:
verts_utm = G.apply(M, mesh.V)
origin_utm = np.array([float(np.mean(verts_utm[:, 0])),
                       float(np.mean(verts_utm[:, 1])), 0.0])
verts_local = verts_utm - origin_utm

# one label per face: plane id for plane members, -1 for unassigned
face_plane = np.full(len(mesh.F), -1, int)
for p in planes:
    face_plane[p["faces"]] = p["plane_id"]

face_class = np.zeros(len(mesh.F), int)      # 0 unassigned
CLASS_ID = {"horizontal": 1, "vertical": 2, "sloped": 3}
for p, g in zip(planes, geoms):
    face_class[p["faces"]] = CLASS_ID[g["orientation"]]

face_object = np.full(len(mesh.F), -1, int)
for i, (t, ) in enumerate([(t, ) for t in terraces]):
    pass
oid = 0
object_names = {}
for t in terraces:
    for sel in t["objects"]:
        face_object[sel] = oid
        object_names[oid] = f"{object_records[oid]['label']}_{oid}"
        oid += 1
print(f"labelled {int((face_plane>=0).sum())} / {len(mesh.F)} faces into {len(planes)} planes")
print(f"labelled {int((face_object>=0).sum())} faces into {oid} objects")

In [ ]:
group_names = {}
for p, g in zip(planes, geoms):
    group_names[p["plane_id"]] = (
        f"plane{p['plane_id']:03d}_{g['orientation']}_"
        f"{'flat' if not np.isfinite(g['grid_azimuth_deg']) else SO._compass(g['true_azimuth_deg'])}"
        f"_{p['area_m2']:.1f}m2")
group_names[-1] = "unassigned"

hdr = (f"generated by rooflib\n"
       f"yaw {solution['yaw_deg']:.4f} deg grid, scale {solution['scale']:.5f}\n"
       f"grid convergence {convergence:+.4f} deg (true = grid + convergence)")

X.write_obj(OUTPUT_DIR / "roof_segmented_local_enu.obj", verts_local, mesh.F,
            groups=face_plane, group_names=group_names,
            header_comment=hdr + f"\nlocal ENU, origin_utm = {origin_utm.tolist()}")
X.write_obj(OUTPUT_DIR / "roof_segmented_utm43n.obj", verts_utm, mesh.F,
            groups=face_plane, group_names=group_names,
            header_comment=hdr + "\nabsolute UTM 43N (EPSG:32643)")
X.write_ply_labelled(OUTPUT_DIR / "roof_classes_local_enu.ply", verts_local, mesh.F, face_class)
print("wrote OBJ (local + UTM) and a class-coloured PLY")

In [ ]:
sidecar = X.transform_sidecar(
    solution, M, origin_utm, dsm.crs_wkt, convergence,
    diagnostics={
        "footprint_iou": check["footprint_iou"],
        "footprint_mesh_m2": check["footprint_mesh_m2"],
        "footprint_target_m2": check["footprint_target_m2"],
        "height_residual_median_m": check["height_resid_median_m"],
        "height_residual_mad_m": check["height_resid_mad_m"],
        "deck_elevation_m": deck_z,
        "dsm_roof_median_m": target["roof_median"],
        "chirality_verdict": hand["verdict"],
        "chirality_margin": hand["margin"],
        "yaw_180_overlap_margin": ambiguity["overlap_margin"],
        "yaw_180_height_margin": ambiguity["hcorr_margin"],
        "refine_cost_start": solution["cost_start"],
        "refine_cost_final": solution["cost"],
    },
    notes=[
        "building_mask overshoots the roof; the alignment target is mask AND DSM>roof-2m.",
        "The 180 deg yaw ambiguity was resolved by superstructure height correlation, "
        "not by footprint overlap, which is near-degenerate on this footprint.",
        "Mesh and DSM disagree on superstructure height by 1.2-1.5 m; the mesh is used "
        "as authoritative for on-roof object geometry and the DSM for georeferencing "
        "and off-building context.",
        "DSM vertical datum is undeclared in the source file.",
    ])
X.save_json(OUTPUT_DIR / "transform.json", sidecar)

X.write_csv(OUTPUT_DIR / "roof_planes.csv", plane_records)
X.write_csv(OUTPUT_DIR / "roof_objects.csv",
            [{**r, "footprint_e_m": r["footprint_bbox_m"][0],
              "footprint_n_m": r["footprint_bbox_m"][1]} for r in object_records],
            columns=["label", "label_confidence", "on_terrace_plane_id", "on_terrace_z",
                     "surface_area_m2", "horizontal_area_m2", "vertical_area_m2",
                     "footprint_e_m", "footprint_n_m", "base_elev_m", "top_elev_m",
                     "height_above_deck_m", "centroid_e", "centroid_n", "rectilinearity"])
X.save_json(OUTPUT_DIR / "shading.json",
            {"horizon_profile_from_deck_centre": horizon,
             "neighbour_sectors": sectors,
             "solar_geometry": sun})
X.save_json(OUTPUT_DIR / "roof_summary.json", {
    "deck_elevation_m": round(deck_z, 3),
    "building_height_above_local_ground_m": round(
        deck_z - float(np.percentile(dsm.data[mask.data != 1], 10)), 2),
    "gross_deck_m2": usable["gross_area_m2"],
    "usable_deck_m2": usable["usable_area_m2"],
    "n_planes": len(planes),
    "n_objects": len(object_records),
    "terraces": [{"plane_id": t["terrace_plane_id"], "z": round(t["terrace_z"], 3),
                  "height_above_deck_m": round(t["terrace_z"] - deck_z, 3),
                  "area_m2": round(t["terrace_area_m2"], 2),
                  "n_objects_on_it": len(t["objects"])} for t in terraces],
    "location": {"latitude": lat, "longitude": lon, "epsg": 32643},
})
print("wrote transform.json, roof_planes.csv, roof_objects.csv, shading.json, roof_summary.json")

In [ ]:
# footprint polygons for GIS
feats = []
for p, g, rec in zip(planes, geoms, plane_records):
    if p["area_m2"] < 1.5:
        continue
    m_, orig_, cell_ = S.footprint_mask(Cw[p["faces"]][:, :2], cell=0.15)
    for poly in X.mask_to_polygons(m_, orig_, cell_, min_area_cells=20):
        feats.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [poly["ring"]]},
            "properties": {k: v for k, v in rec.items()},
        })
X.write_geojson(OUTPUT_DIR / "roof_planes.geojson", feats)
print(f"wrote roof_planes.geojson with {len(feats)} polygons (EPSG:32643)")

print("\n--- outputs ---")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:38s} {f.stat().st_size/1024:9.1f} KB")

## 7b · Site-analysis JSON (target schema)

Emits `site_analysis.json` in the consumer schema: lat/lng polygons, compass
azimuths, `base_height_m` above **ground**, and per-face plane-fit statistics.

Three decisions worth knowing about:

- **Only non-vertical planes are roof faces.** Walls are structure, not roof.
- **Faces are gated on fit quality** (RMSE ≤ 0.20 m, inliers ≥ 0.45). Without it
  the export ships surfaces with 4 m RMSE and 1 % inliers that look authoritative
  and are pure fusion noise. Rejected area is reported in `warnings`.
- **Obstruction `type` is honest.** The schema's vocabulary is
  `vent | ac_unit | chimney | skylight | pipe | unknown`; this roof has water
  tanks, a stair headroom and a terrace block. Forcing those into `vent` or
  `chimney` would be a lie, so `type` is `unknown` and the real classification
  travels alongside in `source_label`.

In [ ]:
ground_elev = float(np.percentile(dsm.data[mask.data != 1], 10))

site = ES.build_site_json(
    address_id="qa-007",                      # <- set your real address id
    planes=planes, geoms=geoms, plane_records=plane_records,
    terraces=terraces, object_records=object_records,
    verts_enu=verts_utm, faces=mesh.F, centroids=Cw,
    dsm=dsm, mask=mask, crs_wkt=dsm.crs_wkt, to_lonlat=G.to_lonlat,
    ground_elev_m=ground_elev, deck_z=deck_z,
    footprint_mask_fn=S.footprint_mask,
    imagery_date=None,                        # not recorded in the source files
    dsm_resolution_m=float(dsm.res[0]),
    provider="iphone-arkit-lidar + photogrammetric-dsm",
    extra_warnings=[
        "mesh_dsm_superstructure_height_disagreement_1.2_1.5m",
        f"scan_incomplete: mesh footprint {check['footprint_mesh_m2']:.1f} m2 "
        f"vs dsm building surface {check['footprint_target_m2']:.1f} m2",
        "dsm_vertical_datum_undeclared",
    ])

X.save_json(OUTPUT_DIR / "site_analysis.json", site)
print(f"ground datum {ground_elev:.2f} m; deck sits {deck_z-ground_elev:.2f} m above it")
print(f"roof_faces {len(site['roof_faces'])}   obstructions {len(site['obstructions'])}")

In [ ]:
print(f"{'id':<9}{'area':>7}{'tilt':>7}{'azim':>8}{'base_h':>8}{'rmse':>7}{'inlier':>8}{'conf':>7}  ring")
for f in site["roof_faces"]:
    az = "flat" if f["azimuth_deg"] is None else f"{f['azimuth_deg']:.1f}"
    print(f"{f['id']:<9}{f['area_m2']:7.2f}{f['tilt_deg']:7.1f}{az:>8}{f['base_height_m']:8.2f}"
          f"{f['plane_fit']['rmse_m']:7.3f}{f['plane_fit']['inlier_ratio']:8.2f}"
          f"{f['confidence']:7.2f}  {len(f['polygon'])} pts")

print(f"\n{'id':<8}{'type':<9}{'source_label':<23}{'w x l':>12}{'h':>7}{'on_face':>9}{'conf':>6}")
for o in site["obstructions"]:
    wl = f"{o['footprint_m']['width']:.1f} x {o['footprint_m']['length']:.1f}"
    print(f"{o['id']:<8}{o['type']:<9}{o['source_label']:<23}{wl:>12}{o['height_m']:7.2f}"
          f"{str(o['on_face']):>9}{o['confidence']:6.2f}")

print("\nwarnings:")
for w in site["warnings"]:
    print("  -", w)

## 8 · Summary

In [ ]:
print("="*74)
print("GEOREFERENCING")
print(f"  yaw {solution['yaw_deg']:.2f} deg grid / {(solution['yaw_deg']+convergence)%360:.2f} deg true")
print(f"  scale {solution['scale']:.4f}   origin UTM43N ({origin_utm[0]:.2f}, {origin_utm[1]:.2f})")
print(f"  footprint IoU {check['footprint_iou']:.3f}   deck {deck_z:.2f} m vs DSM {target['roof_median']:.2f} m")
print(f"  chirality {hand['verdict']}; 180 deg resolved with height margin {ambiguity['hcorr_margin']:.2f}")
print("\nSEGMENTATION")
print(f"  {len(planes)} planes covering {100*Aw[assigned].sum()/Aw.sum():.0f}% of {Aw.sum():.0f} m^2")
print(f"  {len(terraces)} horizontal terraces, {len(object_records)} objects")
print("\nSOLAR")
print(f"  usable deck {usable['usable_area_m2']:.1f} m^2 of {usable['gross_area_m2']:.1f} m^2 gross")
print(f"  optimal orientation: true azimuth {sun['optimal_azimuth_true_deg']:.0f} deg, "
      f"tilt ~{sun['optimal_fixed_tilt_deg_rule_of_thumb']:.0f} deg")
print(f"  dominant shading is the on-roof block ({worst['horizon_elev_deg']:.0f} deg), not neighbours "
      f"({far['elev_angle_deg']:.1f} deg)")
print("="*74)

### Known limitations

Worth reading before trusting a number downstream.

- **Superstructure height disagreement.** The mesh puts the tallest roof structure
  ~5.0 m above the deck; the DSM says ~2.9 m, and the DSM's maximum across the
  whole 100 × 100 m scene is only ~3.5 m above the deck. The mesh is treated as
  authoritative for on-roof geometry (its deck registers to the DSM within ~0.1 m,
  and photogrammetric DSMs systematically under-height small elevated structures),
  but this is **unresolved** and directly affects self-shading estimates.
- **The scan is incomplete.** Mesh footprint ≈ 144 m² against a DSM building
  surface of ≈ 150 m²; the far end of the long axis appears unscanned, so roof
  area is a slight under-estimate.
- **Object labels are heuristic**, with confidences ≤ 0.6. Trust the geometry
  (footprint, height, elevation) far more than the noun attached to it.
- **Vertical datum is undeclared** in `dsm.tif`. Elevations are internally
  consistent and pass through unchanged; they are not tied to a known datum.
  Irrelevant to tilt, azimuth and yield.
- **~16 % of mesh area stays unassigned** to any plane — mostly fusion noise at
  creases. It is retained in the exported geometry, labelled `unassigned`.
- **Scale is fitted to ~1.006** but the IoU curve is flat between 1.02 and 1.04;
  treat absolute dimensions as ±2 %.